# Amharic Speech-to-Speech RAG: ASR → RAG → Gemma 4 E4B (llama.cpp) → TTS, in a plain HTML widget

The complete speech-to-speech pipeline on one Kaggle **GPU T4 x2** runtime. The Gradio demo is gone: the UI is the raw HTML / CSS / JavaScript recorder from the Hohe ASR + Gemma + OmniVoice notebook, extended so the browser plays the answer **sentence by sentence while it is still being generated**.

1. **ASR** `snapwre/hohe-asr-amharic` turns the recording into text.
2. **Retrieval** (your FAISS + character-n-gram TF-IDF hybrid index) finds the 3 best chunks.
3. **LLM** `gemma-4-E4B-it` (GGUF, served by **llama.cpp `llama-server`**) answers from those chunks and streams the answer.
4. **TTS** `african-low-resource/omnivoice-amharic` speaks each sentence **as soon as it is complete**.

**Why it is faster than the Gradio version**
- No Gradio server or queue. The widget is a plain page served by a tiny web server inside the notebook and reached through a tunnel link; it talks to Python with two small HTTP calls.
- The browser sends the recorded 16 kHz WAV and Python decodes it **in memory** (no temp file, no resampling).
- Each sentence's audio is pushed to the browser the moment it is synthesized and queued gapless with Web Audio. Nothing is written to disk on the widget path.
- The LLM stream is read by its own thread, so the stage timings are not distorted by TTS work.

**What the widget shows (every run)**
- Cards: **ASR**, **Retrieval**, **LLM → 1st sentence**, **TTS → 1st audio**, **TTFS (server)** and **TTFS (browser)**. The first four add up to the server TTFS.
- A **timeline** with the time each stage handed over to the next, plus the compute detail of each stage (ASR decode/features/model, retrieval compute, LLM prefill and tok/s, TTS synth time, audio length, RTF).
- A **browser table**: encode and upload, delivery back, and the true end-to-end **time to first sentence heard** (Stop click → audio audible).
- A **per-sentence table** with LLM-ready time, wait for TTS, TTS time, RTF, audio-ready time and heard time.

**Two run modes** (checkbox in the widget)
- **Streaming (default):** each sentence from the LLM goes to TTS the moment `።`, `፤`, `?` or `!` arrives (the first phrase may also end at `፣`).
- **Single-pass:** the full LLM answer, then one TTS call (baseline for comparison).

**Needs in the attached dataset (`amharic_rag/`)**: `src/normalize.py`, the `index/` folder, `data/questions_mapped.json`. `system_prompt.txt` is created on first run and re-read on every request.

**First run:** the prebuilt `llama-server` from the dataset is used (compiled from source only if it cannot start). Run the cells from top to bottom.

**Sections**: 1 Setup · 2 Configuration · 3 HF auth · 4 ASR · 5 TTS · 6 Retrieval · 7 llama.cpp server · 8 LLM helpers · 9 Pipeline · 10 Voice widget · 11 Stop

**Run:** accelerator **GPU T4 x2**, Internet on, dataset `amharic-rag-project` attached → Run all (about 15 min). Section 10 prints a link and then keeps running: leave that cell running and open the link in a new browser tab to use the voice widget (the notebook's own output frame blocks the microphone).

Speed-ups, each switched by `IMPROVE` in section 2:

| Switch | What it does |
|---|---|
| `second_gpu` | Gemma runs on Kaggle's second T4; ASR, TTS and retrieval keep the first |
| `router` | greetings, thanks, goodbye, "who are you": answered with audio made at start-up |
| `answer_cache` | a question asked before is answered from memory (same words, or e5 similarity ≥ 0.98) |
| `numbers_to_words` | digits in Gemma's answer are turned into Amharic words before TTS |
| `early_first_words` | (off by default) the first N words of an answer go to TTS before the sentence ends |
| `swa_full` | (off by default) llama-server keeps the system prompt cached |


## 1. Environment setup

Use a **T4 GPU** runtime. `transformers` stays pinned to 5.18.0 because the ASR and TTS models were set up with it; the LLM no longer uses `transformers` at all.

In [1]:
# both T4s stay visible: PyTorch uses the first, llama-server is told which one to use (see start_server)
import os

In [2]:
!nvidia-smi
!pip install -q "transformers==5.18.0" accelerate sentencepiece librosa soundfile pydub tensorboardX webdataset pandas faiss-cpu sentence-transformers scikit-learn pymupdf requests
!pip install -q --no-deps omnivoice==0.2.1

Tue Oct  6 19:45:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import gc, json, pickle, sys, re, os, time, uuid, shutil, subprocess, atexit, tempfile
import io, base64, math, queue, threading
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import torch
import transformers
from IPython.display import display, Audio

print("Torch:", torch.__version__)
print("Transformers (must be 5.18.0):", transformers.__version__)
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU."

DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


def vram_report(tag=""):
    """Memory held by the PyTorch models in this notebook (ASR, TTS, retrieval)."""
    alloc = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM {tag}: {alloc:.2f} GB allocated (PyTorch) / {total:.2f} GB total")


def gpu_used_gb():
    """Whole-GPU memory in use, including the llama-server process."""
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9


def gpu_report(tag=""):
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU {tag}: {gpu_used_gb():.2f} GB in use (all processes) / {total:.2f} GB total")


def free_cuda():
    gc.collect()
    torch.cuda.empty_cache()


class Timer:
    """Wall-clock timer that synchronizes CUDA on entry and exit."""

    def __enter__(self):
        torch.cuda.synchronize()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        torch.cuda.synchronize()
        self.elapsed = time.perf_counter() - self._t0
        return False

Torch: 2.11.0+cu128
Transformers (must be 5.18.0): 5.18.0
GPU: Tesla T4
VRAM total: 15.64 GB


In [4]:
# [Kaggle] no Google Drive: copy the corrected project from the attached dataset to a writable folder
import glob as _glob
import pathlib

WRITABLE_DIR = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle/working") else tempfile.gettempdir())
_matches = sorted(_glob.glob("/kaggle/input/**/src/normalize.py", recursive=True))
if _matches:
    _src = os.path.dirname(os.path.dirname(_matches[0]))
    PROJECT = str(WRITABLE_DIR / "amharic_rag")
    shutil.copytree(_src, PROJECT, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("index_e5base", "voice_ref_male.wav"))
else:
    PROJECT = str(WRITABLE_DIR / "amharic_rag")

INDEX_DIR = f"{PROJECT}/index"
DATA = f"{PROJECT}/data"
sys.path.insert(0, PROJECT)                       # so "from src.normalize import ..." works

# Preflight: fail early with a clear message if something is missing.
required = [
    f"{PROJECT}/src/normalize.py",
    f"{INDEX_DIR}/chunks.faiss", f"{INDEX_DIR}/embeddings.npy",
    f"{INDEX_DIR}/chunks.json", f"{INDEX_DIR}/build_info.json",
    f"{DATA}/questions_mapped.json",
]
missing = [p for p in required if not os.path.exists(p)]
assert not missing, "Missing files in Dataset / Drive:\n" + "\n".join(missing)
print(f"All required files found in {PROJECT}.")


All required files found in /kaggle/working/amharic_rag.


## 2. Configuration

All tunable values live here.

In [5]:
# ---- Models ------------------------------------------------------------------
ASR_MODEL_ID = "snapwre/hohe-asr-amharic"
TTS_MODEL_ID = "african-low-resource/omnivoice-amharic"

# ---- ASR ---------------------------------------------------------------------
ASR_SAMPLE_RATE = 16000
ASR_DTYPE = torch.float16          # set torch.float32 if transcripts degrade

# ---- LLM: llama.cpp server ------------------------------------------------------
# ggml-org only publishes Q4_0 / Q8_0 for this model, so Q4_K_M is searched in the other repos first.
LLAMA_REPOS = [
    "bartowski/google_gemma-4-E4B-it-GGUF",
    "unsloth/gemma-4-E4B-it-GGUF",
    "ggml-org/gemma-4-E4B-it-GGUF",
]
LLAMA_QUANT = "Q4_K_M"             # if Amharic quality drops, try "Q8_0" (about 8 GB, slower but fits the T4)
LLAMA_QUANT_FALLBACKS = ["Q4_0", "Q8_0"]   # used only if LLAMA_QUANT is found in none of the repos
LLAMA_CTX = 4096                   # prompt + answer must fit; ~1000 prompt tokens are typical here

import socket
def _free_port():
    # A random free port: Colab itself uses some low ports (8080 among them), so never hard-code one.
    with socket.socket() as s_:
        s_.bind(("127.0.0.1", 0))
        return s_.getsockname()[1]

SERVER_PORT = _free_port()
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"

# ---- LLM decoding ----------------------------------------------------------------
LLM_MAX_NEW_TOKENS = 200           # answers are meant to be 1-2 sentences
LLM_TEMPERATURE = 0.0              # greedy = reproducible measurements
LLM_REPEAT_PENALTY = 1.05

# ---- Retrieval -------------------------------------------------------------------
RETRIEVAL_K = 3                    # chunks passed to the LLM

# ---- Sentence streaming (what TTS receives) -------------------------------------------
PHRASE_ENDINGS = ("፣", "።", "፤", ".", "!", "?")   # allowed to end the FIRST phrase (lowest latency)
STRONG_ENDINGS = ("።", "፤", "፧", "!", "?")         # later chunks wait for a real sentence end
FIRST_PHRASE_MIN_WORDS = 2

# ---- TTS -----------------------------------------------------------------------------
TTS_STEPS_FIRST = 8               # steps for the FIRST sentence (speed). Set equal to TTS_STEPS_FULL for max quality
TTS_STEPS_FULL = 16               # steps for later sentences / single-pass answers; raise toward 32 for quality
TTS_GUIDANCE = 2.0
TTS_SEED = 42                      # fixed seed keeps the auto-voice consistent across chunks
TTS_REFINE_FINAL = False           # True: after streaming, re-synthesize the whole answer at TTS_STEPS_FULL for the
                                   # final audio file (costs extra compute; does not change first_audio_ready)
CHUNK_GAP_S = 0.12                 # silence inserted between sentence chunks in the final audio file

FALLBACK_REPLY = "ሰላም! እንዴት ልረዳህ እችላለሁ?"

# ---- Voice widget (section 10) -------------------------------------------------------
WIDGET_STREAMING_DEFAULT = True    # initial state of the "Streaming" checkbox in the widget
LONG_POLL_S = 0.4                  # the browser waits up to this long for new events, and gets them the moment they exist

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="s2s_"))
print("Output directory:", OUTPUT_DIR)
print("llama-server port:", SERVER_PORT)

Output directory: /tmp/s2s_0v9ojky3
llama-server port: 36381


### Switches

In [6]:
# each one can be switched off
IMPROVE = {
    "second_gpu": True,          # Gemma on Kaggle's second T4 (leaves GPU0 for ASR + TTS)
    "router": True,              # Small talk (greetings, thanks, "who are you") answered without calling the LLM
    "answer_cache": True,        # Exact and high-similarity questions answered from memory
    "cache_min": 0.98,           # Minimum similarity (dense embedding cosine) to trust the cached answer
    "numbers_to_words": True,    # LLM digits ("60") turned into Amharic words ("ስልሳ") before TTS
    "swa_full": False,           # Off by default: saves only 0.08 s on Kaggle and changes the wording of answers
    "early_first_words": 0,      # Off by default: 0.5 s faster, but triples garbled speech
}

def llm_gpu():
    # Only use GPU 1 if at least 2 physical GPUs are allocated
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if IMPROVE.get("second_gpu", True) and num_gpus >= 2:
        return 1
    return 0


## 3. Hugging Face authentication (optional)

The models used here are public, but a token avoids download rate limits.

In [7]:
import os

# HF_TOKEN must exist as a Python variable (section 7 uses it) AND as an environment variable
# (so transformers / sentence-transformers / omnivoice also download authenticated).
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or None
except Exception as e:
    print(f"Kaggle secret HF_TOKEN not available ({type(e).__name__}).")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("Using HF token (variable + environment set)")
else:
    print("No HF_TOKEN found. Continuing anonymously.")


Kaggle secret HF_TOKEN not available (ConnectionError).
No HF_TOKEN found. Continuing anonymously.


## 4. Hohe Amharic ASR

In [8]:
from transformers import AutoProcessor, AutoModelForCTC

free_cuda()

asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_ID)
asr_model = AutoModelForCTC.from_pretrained(
    ASR_MODEL_ID,
    dtype=ASR_DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE).eval()

print("ASR loaded:", next(asr_model.parameters()).dtype)
vram_report("after ASR")


def _asr_inputs_to_gpu(inputs):
    return {
        k: v.to(DEVICE, dtype=ASR_DTYPE) if v.is_floating_point() else v.to(DEVICE)
        for k, v in inputs.items()
    }


def clean_asr_text(text):
    return re.sub(r"\[AMH\]", "", text).strip()


def _asr_run(audio, t_start, load_s):
    """Features -> CTC model -> text for a mono 16 kHz float32 waveform. Returns the cleaned transcript and timings."""
    t1 = time.perf_counter()
    inputs = asr_processor(audio, sampling_rate=ASR_SAMPLE_RATE, return_tensors="pt")
    gpu_inputs = _asr_inputs_to_gpu(inputs)
    prep_s = time.perf_counter() - t1

    with Timer() as t, torch.inference_mode():
        logits = asr_model(**gpu_inputs).logits

    pred_ids = logits.argmax(dim=-1)
    raw = asr_processor.batch_decode(pred_ids)[0]

    return {
        "raw": raw,
        "transcript": clean_asr_text(raw),
        "load_s": load_s,                 # reading / decoding the audio
        "preprocessing_s": prep_s,        # feature extraction + copy to GPU
        "inference_s": t.elapsed,         # the CTC model itself
        "total_s": time.perf_counter() - t_start,
    }


def transcribe(audio_path):
    """From a file on disk."""
    t_start = time.perf_counter()
    audio, _ = librosa.load(audio_path, sr=ASR_SAMPLE_RATE, mono=True)
    return _asr_run(audio, t_start, time.perf_counter() - t_start)


def transcribe_wav_bytes(wav_bytes):
    """From the WAV the browser recorded (used by the widget): no temp file, and no resampling when it is already 16 kHz."""
    t_start = time.perf_counter()
    audio, sr = sf.read(io.BytesIO(wav_bytes), dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != ASR_SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=ASR_SAMPLE_RATE)
    return _asr_run(np.ascontiguousarray(audio), t_start, time.perf_counter() - t_start)


# Warm-up so the first real request is not slowed by CUDA initialization.
_dummy = asr_processor(np.zeros(ASR_SAMPLE_RATE, dtype=np.float32),
                       sampling_rate=ASR_SAMPLE_RATE, return_tensors="pt")
with torch.inference_mode():
    _ = asr_model(**_asr_inputs_to_gpu(_dummy)).logits
torch.cuda.synchronize()
print("ASR warm-up complete")

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/32.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/797 [00:00<?, ?it/s]

ASR loaded: torch.float16
VRAM after ASR: 1.21 GB allocated (PyTorch) / 15.64 GB total
ASR warm-up complete


## 5. OmniVoice Amharic TTS

In [9]:
from omnivoice import OmniVoice

try:
    from omnivoice import OmniVoiceGenerationConfig
except ImportError:
    from omnivoice.models.omnivoice import OmniVoiceGenerationConfig

free_cuda()
tts_model = OmniVoice.from_pretrained(TTS_MODEL_ID, device_map="cuda:0", dtype=torch.float16)
tts_model.eval()
TTS_SAMPLE_RATE = tts_model.sampling_rate or 24000

print(f"OmniVoice loaded (sampling rate {TTS_SAMPLE_RATE} Hz)")
vram_report("after TTS")

# Pinned once by the "pick a voice" cells below (create_voice_clone_prompt) so every synthesize()
# call clones the same reference instead of letting OmniVoice auto-pick a voice each time, which is
# why the voice used to change between runs. None here = unchanged auto-pick behavior until set.
VOICE_CLONE_PROMPT = None


def synthesize(text, num_step, tag, save=True):
    """Synthesizes text to a WAV file and returns the path, waveform and timing/RTF."""
    text = text.strip()
    if not text:
        raise ValueError("Empty text passed to TTS.")

    config = OmniVoiceGenerationConfig(num_step=num_step, guidance_scale=TTS_GUIDANCE)
    torch.manual_seed(TTS_SEED)

    with Timer() as t, torch.inference_mode():
        audio = tts_model.generate(text=text, language="Amharic", generation_config=config,
                                    voice_clone_prompt=VOICE_CLONE_PROMPT)

    wave = audio[0]
    if isinstance(wave, torch.Tensor):
        wave = wave.detach().cpu().float().numpy()
    wave = np.asarray(wave).squeeze()

    path = None
    if save:   # the widget sends audio straight from memory, so it skips the disk write
        path = OUTPUT_DIR / f"{tag}_{uuid.uuid4().hex[:8]}.wav"
        sf.write(str(path), wave, TTS_SAMPLE_RATE)

    duration = len(wave) / TTS_SAMPLE_RATE
    return {
        "path": str(path) if path else None,
        "waveform": wave,
        "time": t.elapsed,
        "duration": duration,
        "rtf": t.elapsed / duration if duration > 0 else float("nan"),
    }


_ = synthesize("ሰላም", TTS_STEPS_FIRST, "warmup")
print("TTS warm-up complete")
vram_report("all models loaded")

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

OmniVoice loaded (sampling rate 24000 Hz)
VRAM after TTS: 3.25 GB allocated (PyTorch) / 15.64 GB total
TTS warm-up complete
VRAM all models loaded: 3.25 GB allocated (PyTorch) / 15.64 GB total


In [10]:
# ---- Pick a constant (male) voice: audition a few seeds in "auto" mode -----------------------
# OmniVoice.generate() with no reference picks a voice itself each call - this is why the voice
# changes between runs. Listen to each of these and note which seed sounds like a man; the next
# cell locks that one in permanently via voice cloning.
_REF_TEXT = "ሰላም፣ እኔ ረዳትዎ ነኝ፣ ስለ ኢትዮጵያ የሠራተኛ ሕግ ጥያቄዎችን እመልሳለሁ።"
for _seed in [1, 7, 42, 123]:
    torch.manual_seed(_seed)
    _a = tts_model.generate(text=_REF_TEXT, language="Amharic",
                             generation_config=OmniVoiceGenerationConfig(num_step=TTS_STEPS_FULL, guidance_scale=TTS_GUIDANCE))
    _w = _a[0].detach().cpu().float().numpy() if hasattr(_a[0], "detach") else _a[0]
    print(f"seed={_seed}")
    display(Audio(np.asarray(_w).squeeze(), rate=TTS_SAMPLE_RATE))


seed=1


seed=7


seed=42


seed=123


In [11]:
# ---- Lock in the chosen voice ----------------------------------------------------------------
# Default: seed 42. If it doesn't sound right, listen to the candidates in the cell above, set
# REF_SEED to whichever did, and just re-run this one cell - nothing else needs to change.
REF_SEED = 42

torch.manual_seed(REF_SEED)
_ref_audio = tts_model.generate(text=_REF_TEXT, language="Amharic",
                                 generation_config=OmniVoiceGenerationConfig(num_step=TTS_STEPS_FULL, guidance_scale=TTS_GUIDANCE))
_ref_wave = _ref_audio[0].detach().cpu().float().numpy() if hasattr(_ref_audio[0], "detach") else _ref_audio[0]
VOICE_REF_PATH = str(WRITABLE_DIR / "voice_ref_male.wav")
sf.write(VOICE_REF_PATH, np.asarray(_ref_wave).squeeze(), TTS_SAMPLE_RATE)

# ref_text is supplied directly (the exact phrase just generated from), so this skips
# omnivoice's auto-transcribe fallback, which would otherwise lazy-load a whisper-large-v3-turbo
# model just to caption the reference clip.
VOICE_CLONE_PROMPT = tts_model.create_voice_clone_prompt(ref_audio=VOICE_REF_PATH, ref_text=_REF_TEXT)
print(f"Voice locked in (seed {REF_SEED}) - every synthesize() call from here on clones this voice.")
display(Audio(VOICE_REF_PATH))


Voice locked in (seed 42) - every synthesize() call from here on clones this voice.


## 6. Retrieval

Loads the saved index (it stops if index and chunks are out of sync), builds the character n-gram keyword index, and defines `retrieve()`. Unchanged from your working notebook.

In [12]:
import sys, os, json, time, pickle
import numpy as np
import faiss
import torch

from src.normalize import normalize_query          # same function used on documents

with open(f"{INDEX_DIR}/build_info.json", encoding="utf-8") as f:
    info = json.load(f)
with open(f"{INDEX_DIR}/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
emb = np.load(f"{INDEX_DIR}/embeddings.npy")            # (N, 384) normalized
index = faiss.read_index(f"{INDEX_DIR}/chunks.faiss")
assert index.ntotal == len(chunks) == emb.shape[0] == info["num_chunks"], "index and chunks out of sync"

from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(info["model"], device=device)
model.max_seq_length = 512
print("device:", device, "| chunks:", len(chunks))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

device: cuda | chunks: 227


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# char_wb = n-grams inside word boundaries; 2-4 chars suits Amharic syllable (fidel) units
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True, min_df=1)
doc_texts = [c["text_folded"] for c in chunks]
kw_matrix = tfidf.fit_transform(doc_texts)               # sparse (N, features), L2-normalized rows
print("keyword features:", kw_matrix.shape[1])

with open(f"{INDEX_DIR}/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

keyword features: 22280


In [14]:
def _minmax(x):
    lo, hi = float(x.min()), float(x.max())
    return (x - lo) / (hi - lo) if hi - lo > 1e-9 else np.zeros_like(x)

def retrieve(query, k=3, alpha=0.6, mode="hybrid"):
    """
    mode: "hybrid" (default), "dense" or "keyword" (the last two are for comparison in step 6).
    alpha: weight of the dense score in hybrid mode (1 - alpha goes to the keyword score).
    Returns (results, timing_ms). Each result has id, score, dense, keyword, text_spoken, text_clean.
    """
    t0 = time.perf_counter()

    q_norm = normalize_query(query)                        # fold + numbers to words (same as documents)
    t1 = time.perf_counter()

    dense = np.zeros(len(chunks), dtype="float32")
    if mode in ("hybrid", "dense"):
        qv = model.encode(["query: " + q_norm], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
        dense = (emb @ qv[0])                              # cosine similarities, all chunks (exact)
    t2 = time.perf_counter()

    kw = np.zeros(len(chunks), dtype="float32")
    if mode in ("hybrid", "keyword"):
        qk = tfidf.transform([q_norm])
        kw = (kw_matrix @ qk.T).toarray().ravel().astype("float32")
    t3 = time.perf_counter()

    if mode == "dense":
        score = dense
    elif mode == "keyword":
        score = kw
    else:
        score = alpha * _minmax(dense) + (1 - alpha) * _minmax(kw)

    top = np.argsort(-score)[:k]
    results = [{
        "id": chunks[i]["id"],
        "score": float(score[i]),
        "dense": float(dense[i]),        # raw cosine: keep for the "I don't know" threshold in step 8
        "keyword": float(kw[i]),
        "text_spoken": chunks[i]["text_spoken"],
        "text_clean": chunks[i]["text_clean"],
    } for i in top]
    t4 = time.perf_counter()

    timing = {
        "normalize_ms": (t1 - t0) * 1000,
        "embed_ms": (t2 - t1) * 1000,
        "keyword_ms": (t3 - t2) * 1000,
        "rank_ms": (t4 - t3) * 1000,
        "total_ms": (t4 - t0) * 1000,
    }
    return results, timing

# warm-up (first call is slow because of CUDA / lazy init; do not time it)
for _ in range(3):
    retrieve("የሙከራ ጊዜ ስንት ነው?")

In [15]:
with open(f"{DATA}/questions_mapped.json", encoding="utf-8") as f:
    qs = json.load(f)
qs = qs["questions"] if isinstance(qs, dict) else qs
print("questions:", len(qs), "| answerable:", sum(1 for q in qs if q.get("answerable")))

res, tm = retrieve(qs[0]["question"])
print(f"retrieve() total {tm['total_ms']:.1f} ms on {device}")

questions: 43 | answerable: 38
retrieve() total 16.2 ms on cuda


## 7. llama.cpp: build (once), download the GGUF, start the server

- The `llama-server` binary is compiled for the T4 only and cached to `amharic_rag/bin/` on Drive. If the cached binary does not run in a later session, it is rebuilt automatically.
- The GGUF goes to Colab's local disk each session.
- The server uses all-GPU layers, one slot and a **random free port**. It is only ever stopped by its own process, never by killing a port (Colab uses some ports itself).

In [16]:
# [Kaggle] Set up a llama-server binary that supports Gemma 4 and actually starts on this machine
# Cause of the "libcudart.so.13: cannot open shared object file" failure: the pre-shipped binary is built against CUDA 13,
# Kaggle has CUDA 12, and the old check passed anyway because the word "llama" appears in the loader's own error message.
import os, sys, shutil, subprocess, pathlib, time, glob

WRITABLE_DIR = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle/working") else tempfile.gettempdir())
LOCAL_BIN = str(WRITABLE_DIR / "llama-server")
CUDA13_DIR = WRITABLE_DIR / "cuda13_libs"          # CUDA 13 runtime libs go here, so torch's CUDA 12 stays untouched

# Kill any existing background instance to prevent Linux [Errno 26] Text file busy
subprocess.run(["pkill", "-9", "-f", "llama-server"], check=False)
time.sleep(0.5)

def _add_lib_dir(d):
    d = str(d)
    cur = os.environ.get("LD_LIBRARY_PATH", "")
    if d not in cur.split(":"):
        os.environ["LD_LIBRARY_PATH"] = d + (":" + cur if cur else "")

for _d in ["/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib", "/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu"]:
    _add_lib_dir(_d)
os.environ["LIBRARY_PATH"] = ":".join(["/usr/local/cuda/lib64/stubs", "/usr/local/cuda/lib64", os.environ.get("LIBRARY_PATH", "")])
os.environ["PATH"] = f"/usr/local/cuda/bin:{os.environ.get('PATH', '')}"
for _d in glob.glob(str(CUDA13_DIR / "nvidia" / "*" / "lib")):     # already installed earlier? make the loader see them
    _add_lib_dir(_d)

# Ensure CUDA stub driver symlink exists for any CMake/linking needs
try:
    if os.path.exists("/usr/local/cuda/lib64/stubs/libcuda.so") and not os.path.exists("/usr/local/cuda/lib64/stubs/libcuda.so.1"):
        os.symlink("/usr/local/cuda/lib64/stubs/libcuda.so", "/usr/local/cuda/lib64/stubs/libcuda.so.1")
except Exception:
    pass

def _missing_libs(path):
    """Shared libraries the binary needs but the loader cannot find."""
    r = subprocess.run(["ldd", path], capture_output=True, text=True, env=os.environ)
    return [ln.split("=>")[0].strip() for ln in r.stdout.splitlines() if "not found" in ln]

def _bin_works(path):
    """True only if the binary really starts (no missing libraries, --version exits cleanly)."""
    if not os.path.exists(path):
        return False
    try:
        os.chmod(path, 0o755)
        if _missing_libs(path):
            return False
        r = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=60, env=os.environ)
    except Exception:
        return False
    out = (r.stdout + r.stderr).lower()
    return r.returncode == 0 and "error while loading shared libraries" not in out

def _safe_copy(src, dst):
    try:
        if os.path.exists(dst):
            os.remove(dst)
    except Exception:
        pass
    shutil.copy(src, dst)
    os.chmod(dst, 0o755)

def _install_cuda13_libs():
    print("Installing CUDA 13 runtime libraries (libcudart.so.13, libcublas.so.13) into", CUDA13_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", str(CUDA13_DIR),
                    "nvidia-cuda-runtime", "nvidia-cublas"], check=True)
    for d in glob.glob(str(CUDA13_DIR / "nvidia" / "*" / "lib")):
        _add_lib_dir(d)

# 1. Reuse the binary from an earlier run of this cell if it starts
ready = _bin_works(LOCAL_BIN)
if ready:
    print("llama-server already in place and starts correctly:", LOCAL_BIN)

# 2. Pre-shipped binary from the dataset (built for the T4, sm_75, Gemma 4 support)
if not ready:
    candidate_bins = [f"{PROJECT}/bin/llama-server-sm75", f"{PROJECT}/bin/llama-server"] + \
        glob.glob("/kaggle/input/**/llama-server*", recursive=True)
    for cand in candidate_bins:
        if not (os.path.isfile(cand) and os.path.getsize(cand) > 10_000_000):
            continue
        print("Found pre-shipped llama-server:", cand)
        _safe_copy(cand, LOCAL_BIN)
        miss = _missing_libs(LOCAL_BIN)
        if miss:
            print("  missing shared libraries:", miss)
            if any(m.endswith(".13") for m in miss):
                try:
                    _install_cuda13_libs()
                except Exception as e:
                    print("  could not install the CUDA 13 libraries:", e)
        if _bin_works(LOCAL_BIN):
            print("  binary starts correctly.")
            ready = True
            break
        print("  still missing:", _missing_libs(LOCAL_BIN), "- trying the next option.")

# 3. Fallback: compile llama.cpp against this machine's own CUDA toolkit (about 10-15 minutes, T4 only).
#    (The old b4880 release download was removed: it is a CPU-only build from March 2025 and cannot load Gemma 4.)
if not ready:
    assert shutil.which("nvcc") or os.path.exists("/usr/local/cuda/bin/nvcc"), \
        "No working llama-server in the dataset and no CUDA compiler (nvcc) here to build one. Re-upload a binary that matches this CUDA version."
    print("Building llama.cpp with CUDA from source ...")
    env = os.environ.copy()
    src_dir = str(WRITABLE_DIR / "llama.cpp")
    if not os.path.exists(src_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", src_dir], check=True)
    shutil.rmtree(f"{src_dir}/build", ignore_errors=True)
    subprocess.run(["cmake", "-B", f"{src_dir}/build", "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
                    "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"],
                   cwd=src_dir, env=env, check=True)
    subprocess.run(["cmake", "--build", f"{src_dir}/build", "--config", "Release", f"-j{os.cpu_count() or 4}",
                    "--target", "llama-server"], cwd=src_dir, env=env, check=True)
    _safe_copy(f"{src_dir}/build/bin/llama-server", LOCAL_BIN)

assert _bin_works(LOCAL_BIN), f"llama-server still does not start. Missing libraries: {_missing_libs(LOCAL_BIN)}"
print("llama-server executable verified at:", LOCAL_BIN)


Found pre-shipped llama-server: /kaggle/input/datasets/samuelkeno/amharic-rag-project/amharic_rag/bin/llama-server-sm75
  missing shared libraries: ['libcudart.so.13', 'libcublas.so.13']
Installing CUDA 13 runtime libraries (libcudart.so.13, libcublas.so.13) into /kaggle/working/cuda13_libs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.3/439.3 MB 4.4 MB/s eta 0:00:00
  binary starts correctly.
llama-server executable verified at: /kaggle/working/llama-server


In [17]:
# Verify the llama-server binary runs (stops here with a clear message instead of failing later at server start)
res = subprocess.run([LOCAL_BIN, "--version"], capture_output=True, text=True, env=os.environ)
print((res.stdout + res.stderr).strip())
assert res.returncode == 0, "llama-server does not start; re-run the cell above and read its messages."


version: 0.5.0-dev (build 1, commit 2ca15f5)
built with GNU 13.3.0 for Linux x86_64


In [18]:
from huggingface_hub import HfApi, hf_hub_download

MODEL_DIR = "/kaggle/working/gguf"   # [Kaggle]
os.makedirs(MODEL_DIR, exist_ok=True)

# Works even if the token cell (section 3) was skipped after a kernel restart.
HF_TOKEN = globals().get("HF_TOKEN") or os.environ.get("HF_TOKEN") or None


def _list_files(repo):
    """List repo files; if the token is rejected (401), retry anonymously (the repos are public)."""
    tokens = [HF_TOKEN, None] if HF_TOKEN else [None]
    last = None
    for tok in tokens:
        try:
            return HfApi().list_repo_files(repo, token=tok)
        except Exception as e:
            last = e
    raise last


def find_gguf(quant):
    """Returns (repo, filename) of the first repo that has a main-model GGUF for this quantization."""
    for repo in LLAMA_REPOS:
        try:
            names = _list_files(repo)
        except Exception as e:
            print(f"  skip {repo}: {type(e).__name__}: {str(e)[:150]}")
            continue
        hits = [f for f in names if f.endswith(".gguf") and "mmproj" not in f.lower()
                and quant.lower() in f.lower().replace("-", "_")]
        if hits:
            return repo, sorted(hits, key=len)[0]
        print(f"  {repo}: no {quant} file")
    return None


choice = None
for q in [LLAMA_QUANT] + LLAMA_QUANT_FALLBACKS:
    choice = find_gguf(q)
    if choice:
        USED_QUANT = q
        break
assert choice, "No matching GGUF found in any repo (see the messages above); check LLAMA_REPOS / LLAMA_QUANT / Internet."
repo, fname = choice
if USED_QUANT != LLAMA_QUANT:
    print(f"WARNING: {LLAMA_QUANT} not found anywhere, using {USED_QUANT}.")
print(f"Downloading {fname} from {repo}")
with Timer() as t:
    try:
        MODEL_PATH = hf_hub_download(repo, fname, local_dir=MODEL_DIR, token=HF_TOKEN)
    except Exception as e:
        if HF_TOKEN and "401" in str(e):
            print("Token rejected, retrying anonymously ...")
            MODEL_PATH = hf_hub_download(repo, fname, local_dir=MODEL_DIR, token=None)
        else:
            raise
print(f"{MODEL_PATH}  ({os.path.getsize(MODEL_PATH) / 1e9:.2f} GB, {t.elapsed:.0f}s)")


google_gemma-4-E4B-it-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 5.41GB            

google_gemma-4-E4B-it-Q4_K_M.gguf: downloading bytes:           |  0.00B            

/kaggle/working/gguf/google_gemma-4-E4B-it-Q4_K_M.gguf  (5.41 GB, 14s)


In [19]:
SERVER_LOG = "/kaggle/working/llama_server.log"   # [Kaggle]
server_proc = None


def stop_server():
    global server_proc
    if server_proc is not None and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=15)
        except Exception:
            server_proc.kill()
    server_proc = None


_HELP_CACHE = {}


def _supports_flag(bin_path, flag):
    """CUDA builds can take >5 s to print -h; a timeout used to silently drop --jinja. Cached, 60 s timeout."""
    if bin_path not in _HELP_CACHE:
        try:
            r = subprocess.run([bin_path, "-h"], capture_output=True, text=True, timeout=60, env=os.environ)
            _HELP_CACHE[bin_path] = r.stdout + r.stderr
        except Exception as e:
            print(f"[could not read llama-server -h ({type(e).__name__}); assuming --jinja is supported]")
            return flag == "--jinja"
    return flag in _HELP_CACHE[bin_path]


def start_server(timeout=600):
    global server_proc
    stop_server()
    # Only ever kill our own server, matched by exact process name. Never kill by port: Colab's own
    # services listen on some ports, and killing them disconnects the runtime.
    subprocess.run(["pkill", "-9", "-f", "llama-server"], check=False)
    time.sleep(1)
    
    assert os.path.exists(LOCAL_BIN), f"llama-server binary not found at {LOCAL_BIN}"
    assert os.path.exists(MODEL_PATH), f"GGUF model not found at {MODEL_PATH}"
    
    # Build command safely, only including optional flags if recognized by the binary
    cmd = [
        LOCAL_BIN,
        "-m", MODEL_PATH,
        "-ngl", "99",
        "-c", str(LLAMA_CTX),
        "--host", "127.0.0.1",
        "--port", str(SERVER_PORT),
        "-np", "1",
    ]
    if _supports_flag(LOCAL_BIN, "--jinja"):
        cmd.append("--jinja")
    if IMPROVE.get("swa_full", False) and _supports_flag(LOCAL_BIN, "--swa-full"):
        cmd.append("--swa-full")
        
    log = open(SERVER_LOG, "w")
    target_gpu = str(llm_gpu())
    env = os.environ.copy()
    if target_gpu:
        env["CUDA_VISIBLE_DEVICES"] = target_gpu
        
    print(f"Starting llama-server on port {SERVER_PORT} (GPU {target_gpu})...")
    server_proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < timeout:
        if server_proc.poll() is not None:
            log.flush()
            log_text = open(SERVER_LOG, encoding="utf-8", errors="replace").read()
            print(f"\n[llama-server Error - Process Exited with Code {server_proc.returncode}]")
            print(f"Command executed: {' '.join(cmd)}")
            print("--- Log Output ---")
            print(log_text[-4000:] if log_text.strip() else "(Log file is empty)")
            print("------------------")
            raise RuntimeError(f"llama-server exited with code {server_proc.returncode}; see log details above.")
        try:
            if requests.get(f"{SERVER_URL}/health", timeout=2).status_code == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    else:
        log.flush()
        log_text = open(SERVER_LOG, encoding="utf-8", errors="replace").read()
        print("\n[llama-server Timeout Error]")
        print(log_text[-4000:] if log_text.strip() else "(Log file is empty)")
        raise TimeoutError("llama-server did not become ready in time.")
    print(f"llama-server ready in {time.perf_counter() - t0:.1f}s")


atexit.register(stop_server)
free_cuda()          # hand cached PyTorch memory back before llama.cpp allocates its own
start_server()
gpu_report("with ASR + TTS + retrieval + llama-server loaded")

Starting llama-server on port 36381 (GPU 1)...
llama-server ready in 8.0s
GPU with ASR + TTS + retrieval + llama-server loaded: 4.56 GB in use (all processes) / 15.64 GB total


## 8. LLM helpers: streaming generation, prompt, answering

`generate_stream()` posts to `/v1/chat/completions` with `stream=true` and yields a `sentence` event whenever a phrase is complete, then one `done` event with the timings. If this server build rejects one of the optional request fields, the request is retried automatically with a minimal payload.

In [20]:
def sanitize_to_ethiopic(text):
    """Drops thinking blocks and any non-Ethiopic characters; keeps digits and basic punctuation."""
    text = re.sub(r"<\|channel>.*?<channel\|>", "", text, flags=re.S)
    text = re.sub(r"[^ሀ-፿\s0-9\.\,\?\!]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def split_ready(buf, first):
    """Cuts finished phrases off the front of buf. Returns (phrases, remainder, first_still_pending)."""
    phrases, start = [], 0
    for i, ch in enumerate(buf):
        ends = PHRASE_ENDINGS if first else STRONG_ENDINGS
        if ch == ".":
            is_end = i + 1 < len(buf) and buf[i + 1].isspace()     # "3.5" is not a sentence end
        else:
            is_end = ch in ends
        if not is_end:
            continue
        cand = buf[start:i + 1]
        need = FIRST_PHRASE_MIN_WORDS if first else 1
        if len(sanitize_to_ethiopic(cand).split()) >= need:
            phrases.append(cand)
            start = i + 1
            first = False
    return phrases, buf[start:], first


def _payload(user_text, max_tokens):
    return {
        "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": user_text}],
        "max_tokens": max_tokens,
        "temperature": LLM_TEMPERATURE,
        "repeat_penalty": LLM_REPEAT_PENALTY,
        "stream": True,
        "stream_options": {"include_usage": True},
        "cache_prompt": True,
        "chat_template_kwargs": {"enable_thinking": False},
    }


_WARNED_PAYLOAD = False


def _open_stream(payload):
    """POST the streaming request; if the server rejects an optional field, retry with a minimal payload."""
    global _WARNED_PAYLOAD
    url = f"{SERVER_URL}/v1/chat/completions"
    r = requests.post(url, json=payload, stream=True, timeout=180)
    if r.status_code >= 400:
        body = r.text[:300]
        r.close()
        minimal = {k: v for k, v in payload.items() if k in ("messages", "max_tokens", "temperature", "stream")}
        if not _WARNED_PAYLOAD:
            print(f"[llama-server rejected the request ({r.status_code}: {body}); retrying with a minimal payload]")
            _WARNED_PAYLOAD = True
        r = requests.post(url, json=minimal, stream=True, timeout=180)
        r.raise_for_status()
    return r


def generate_stream(user_text, max_tokens=None):
    """Yields {'type': 'sentence', 'text', 't'} as phrases complete, then one {'type': 'done', ...stats}."""
    max_tokens = max_tokens or LLM_MAX_NEW_TOKENS
    t0 = time.perf_counter()
    buf, raw, n_chunks = "", "", 0
    ttft = first_sentence_t = None
    first = True
    timings, usage = {}, {}

    with _open_stream(_payload(user_text, max_tokens)) as r:
        for line in r.iter_lines(decode_unicode=False):
            if not line:
                continue
            line = line.decode("utf-8")
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            obj = json.loads(data)
            timings = obj.get("timings") or timings
            usage = obj.get("usage") or usage
            for ch in obj.get("choices", []):
                delta = (ch.get("delta") or {}).get("content") or ""
                if not delta:
                    continue
                if ttft is None:
                    ttft = time.perf_counter() - t0
                    yield {"type": "first_token", "t": ttft}
                n_chunks += 1
                raw += delta
                buf += delta
                phrases, buf, first = split_ready(buf, first)
                for p in phrases:
                    p = sanitize_to_ethiopic(p)
                    if not p:
                        continue
                    now = time.perf_counter() - t0
                    first_sentence_t = first_sentence_t if first_sentence_t is not None else now
                    yield {"type": "sentence", "text": p, "t": now}

    rest = sanitize_to_ethiopic(buf)
    total = time.perf_counter() - t0
    if rest:
        first_sentence_t = first_sentence_t if first_sentence_t is not None else total
        yield {"type": "sentence", "text": rest, "t": total}

    new_tokens = int(timings.get("predicted_n") or usage.get("completion_tokens") or n_chunks)
    decode_time = max(total - (ttft or 0.0), 1e-6)
    yield {
        "type": "done",
        "text": sanitize_to_ethiopic(raw),
        "total_s": total,
        "ttft_s": ttft if ttft is not None else total,
        "first_sentence_s": first_sentence_t if first_sentence_t is not None else total,
        "new_tokens": new_tokens,
        "prompt_tokens": int(usage.get("prompt_tokens") or timings.get("prompt_n") or 0),
        "tok_s": float(timings.get("predicted_per_second") or new_tokens / decode_time),
        "prefill_s": float(timings.get("prompt_ms", 0.0)) / 1000.0,
        # server-side time (prefill + decode): unaffected by anything the consumer does between yields
        "server_s": (float(timings.get("prompt_ms", 0.0)) + float(timings.get("predicted_ms", 0.0))) / 1000.0,
    }


def generate_full(user_text, max_tokens=None):
    done = None
    for ev in generate_stream(user_text, max_tokens):
        if ev["type"] == "done":
            done = ev
    return {"text": done["text"], "time": done["total_s"], "new_tokens": done["new_tokens"], **done}


# Warm-up: the first request after startup is slower.
SYSTEM_PROMPT = "You are a helpful assistant that replies in Amharic."
with Timer() as _t:
    _w = generate_full("ሰላም", max_tokens=24)
print(f"warm-up: {_t.elapsed:.2f}s -> {_w['text']}")
with Timer() as _t:
    _w = generate_full("ሰላም", max_tokens=24)
print(f"second call: {_t.elapsed:.2f}s | {_w['new_tokens']} tokens | {_w['tok_s']:.1f} tok/s")

warm-up: 0.50s -> ሰላም! እንዴት ነህሽ?
second call: 0.31s | 12 tokens | 42.8 tok/s


The prompt file lives in the project folder and is re-read on every call. If it still holds the **old default** (the one that allowed 1 to 3 sentences), it is backed up to `system_prompt_v1_backup.txt` and replaced by the shorter version. If you edited it yourself and removed that phrase, it is left alone.

In [21]:
PROMPT_PATH = f"{PROJECT}/system_prompt.txt"

# The sentence the assistant speaks when the context does not contain the answer.
# Have a native speaker check the wording. Keep it short.
IDK_TEXT = "ይቅርታ፣ ለዚህ ጥያቄ መልስ የሚሆን መረጃ የለኝም።"

DEFAULT_PROMPT = f"""You are a voice assistant. Answer the QUESTION using ONLY the CONTEXT passages in the user message.
Reply in Amharic (Ge'ez script) with ONE short sentence, two at most, under 30 words, as if speaking out loud.
Give only the direct answer. Do not repeat the question. Do not add extra explanation and do not cite article numbers unless the question asks for them.
Write numbers as Amharic words, not digits.
Do not use lists, markdown, emojis or English words.
Do not use knowledge from outside the CONTEXT. Do not guess.
If the CONTEXT does not contain the answer, reply with exactly this sentence and nothing else: {IDK_TEXT}
"""

if os.path.exists(PROMPT_PATH):
    _cur = open(PROMPT_PATH, encoding="utf-8").read()
    if "1 to 3 short sentences" in _cur:
        shutil.copy(PROMPT_PATH, f"{PROJECT}/system_prompt_v1_backup.txt")
        open(PROMPT_PATH, "w", encoding="utf-8").write(DEFAULT_PROMPT)
        print("Old default prompt backed up to system_prompt_v1_backup.txt; wrote the shorter prompt.")
    else:
        print("Keeping your existing system_prompt.txt unchanged.")
else:
    open(PROMPT_PATH, "w", encoding="utf-8").write(DEFAULT_PROMPT)
    print("Wrote default prompt to", PROMPT_PATH)


def load_system_prompt():
    with open(PROMPT_PATH, encoding="utf-8") as f:
        return f.read().strip()


SYSTEM_PROMPT = load_system_prompt()
IDK_CLEAN = sanitize_to_ethiopic(IDK_TEXT)

# Optional: your number-to-words function from normalize.py, applied to LLM output before TTS.
NUM2WORDS = None


def prep_for_tts(text):
    return NUM2WORDS(text) if NUM2WORDS else text


def is_idk(text):
    return IDK_CLEAN in sanitize_to_ethiopic(text)


def build_user_message(question, results):
    # text_spoken (numbers already in words) is what the LLM should read.
    # Never pass text_folded: it is for matching only and has lost spelling distinctions.
    context = "\n\n".join(f"[{i + 1}] {r['text_spoken']}" for i, r in enumerate(results))
    return f"CONTEXT:\n{context}\n\nQUESTION: {question}"


MIN_DENSE = None   # set from the score distributions below; None = never refuse on score


def answer_stream(question, k=3, min_dense=None):
    """Retrieve -> optional score gate -> streamed Gemma answer. Yields events; see generate_stream."""
    global SYSTEM_PROMPT
    SYSTEM_PROMPT = load_system_prompt()          # pick up edits to the prompt file

    t0 = time.perf_counter()
    results, _ = retrieve(question, k=k)
    retrieval_s = time.perf_counter() - t0
    top_dense = results[0]["dense"]
    yield {"type": "retrieved", "results": results, "retrieval_s": retrieval_s, "top_dense": top_dense}

    gate = MIN_DENSE if min_dense is None else min_dense
    if gate is not None and top_dense < gate:
        yield {"type": "sentence", "text": IDK_TEXT, "t": 0.0}
        yield {"type": "done", "text": IDK_TEXT, "gated": True, "total_s": 0.0, "ttft_s": 0.0,
               "first_sentence_s": 0.0, "new_tokens": 0, "prompt_tokens": 0, "tok_s": 0.0, "prefill_s": 0.0,
               "server_s": 0.0}
        return

    yield from generate_stream(build_user_message(question, results))


def answer_text(question, k=3, min_dense=None):
    """Same pipeline collected into one dict (used by the smoke test and the evaluation)."""
    out = {"gated": False}
    for ev in answer_stream(question, k, min_dense):
        if ev["type"] == "retrieved":
            out.update(results=ev["results"], retrieval_s=ev["retrieval_s"], top_dense=ev["top_dense"])
        elif ev["type"] == "done":
            out.update(ev)
    out["idk"] = is_idk(out["text"])
    out["llm_s"] = out["total_s"]
    return out

Wrote default prompt to /kaggle/working/amharic_rag/system_prompt.txt


In [22]:
UPDATED_PROMPT = f"""
You are a natural Amharic AI voice assistant specialized in Ethiopia's Labour Proclamation.

CONVERSATIONAL BEHAVIOR:
- You may respond naturally without using CONTEXT only when the user:
  - greets you,
  - thanks you,
  - says goodbye,
  - asks who or what you are,
  - asks what you can do.
- For greetings, thanks, and farewells, respond briefly, warmly and naturally in Amharic.
- If asked who you are, explain naturally that you are an AI assistant designed to answer questions about Ethiopia's Labour Proclamation.
- This conversational exception must NEVER be used to answer factual questions outside the Labour Proclamation.

LEGAL QUESTION RULES:
- For every legal or factual question, answer ONLY using the CONTEXT passages supplied in the user message.
- Never use outside knowledge, memory, assumptions, or guesses.
- If the CONTEXT does not contain enough information to answer, reply exactly:
  {IDK_TEXT}

ANSWER QUALITY:
- Answer the user's actual question directly.
- Be concise but complete.
- Include every legally important value needed to answer the question, but do NOT repeat the wording of the legal provision.
- When several rates, amounts, limits, durations, categories, exceptions, or conditions answer the same question, include all relevant ones compactly.
- State a shared rule only once. Do not repeat the same phrases for every category.
- Use short category descriptions instead of repeating complete legal sentences.
- For questions asking "how", "how much", "how many", "when", "what rate", or similar questions, give the relevant exact values from CONTEXT.
- Do not include unrelated details merely because they appear in the retrieved passages.
- Mention the relevant article when useful.
- Never use information that is not present in CONTEXT.
- Never guess.

SPEAKING STYLE:
- Reply in natural Amharic using Ge'ez script.
- Sound like a helpful conversational voice assistant, not like someone reading a legal document aloud.
- Simple questions should normally use one short sentence.
- Questions with several required facts should normally use no more than two concise sentences.
- Prefer a compact answer of roughly 30 to 60 Amharic words when that is enough to answer completely.
- Do not sacrifice an important legal value merely to meet the preferred length.
- Do not use lists, markdown, emojis, or English words in the spoken answer.
""".strip()

with open(PROMPT_PATH, "w", encoding="utf-8") as f:
    f.write(UPDATED_PROMPT)

SYSTEM_PROMPT = load_system_prompt()

print("Updated system prompt saved.")
print(SYSTEM_PROMPT)

Updated system prompt saved.
You are a natural Amharic AI voice assistant specialized in Ethiopia's Labour Proclamation.

CONVERSATIONAL BEHAVIOR:
- You may respond naturally without using CONTEXT only when the user:
  - greets you,
  - thanks you,
  - says goodbye,
  - asks who or what you are,
  - asks what you can do.
- For greetings, thanks, and farewells, respond briefly, warmly and naturally in Amharic.
- If asked who you are, explain naturally that you are an AI assistant designed to answer questions about Ethiopia's Labour Proclamation.
- This conversational exception must NEVER be used to answer factual questions outside the Labour Proclamation.

LEGAL QUESTION RULES:
- For every legal or factual question, answer ONLY using the CONTEXT passages supplied in the user message.
- Never use outside knowledge, memory, assumptions, or guesses.
- If the CONTEXT does not contain enough information to answer, reply exactly:
  ይቅርታ፣ ለዚህ ጥያቄ መልስ የሚሆን መረጃ የለኝም።

ANSWER QUALITY:
- Answer th

In [23]:
# Digits in the LLM answer would be read badly by TTS. The prompt asks for numbers as words, and you can also
# plug in your number-to-words function from normalize.py here. These are the public functions it offers:
import src.normalize as _norm
print("Functions in src/normalize.py:", [n for n in dir(_norm) if callable(getattr(_norm, n)) and not n.startswith("_")])
# Example (use your real function name, and make sure it only converts numbers, it must NOT fold Fidel):
# from src.normalize import numbers_to_words as NUM2WORDS

Functions in src/normalize.py: ['Path', 'assemble_final_articles', 'clean_document', 'convert_to_spoken', 'extract_amharic_column', 'fold', 'geez_to_int', 'int_to_words', 'main', 'normalize_doc_text', 'normalize_query', 'normalize_text', 'parse_document_structure', 'process_cleaning', 'run_stage7_checks_and_report', 'write_extract_report', 'write_stage2_report', 'write_stage3_report', 'write_stage5_report']


## 9. Pipeline: ASR → retrieval → LLM → TTS

`run_pipeline()` takes the transcript and drives retrieval, the streamed LLM answer and TTS. It records a **timestamp for every hand-over** (`marks`): ASR done, retrieval done, LLM first token, LLM first sentence, first audio ready, all done. The widget turns those into the timeline.

- The LLM stream is read by **its own thread** and every event is stamped when it arrives, so the LLM timings stay accurate even while TTS is busy with sentence 1. `llama-server` keeps generating sentence 2 while TTS works on sentence 1.
- When `emit` is given (the widget), each sentence's audio is sent out as 16-bit WAV straight from memory, with nothing written to disk.
- The "I don't know" reply is synthesized once at start-up, so a refusal costs no TTS time.
- `run_s2s(path)` and `speak_question(text)` are kept for testing without the browser.

In [24]:
# ---- server health ------------------------------------------------------------------------
def ensure_server():
    """Restarts llama-server if it died (for example after a Colab hiccup)."""
    try:
        if requests.get(f"{SERVER_URL}/health", timeout=2).status_code == 200:
            return
    except requests.RequestException:
        pass
    print("llama-server is not responding: restarting it ...")
    start_server()


# ---- cached audio for the "I don't know" reply -----------------------------------------------------
IDK_AUDIO = synthesize(IDK_TEXT, TTS_STEPS_FULL, "idk")
print(f"IDK audio cached ({IDK_AUDIO['duration']:.2f} s)")


def _speak(text, first, save=True):
    """Synthesizes one chunk. The cached audio is reused if the model refused. Returns None for empty text."""
    if is_idk(text):
        a = dict(IDK_AUDIO)
        a.update(time=0.0, rtf=0.0, cached=True, steps=0)
        return a
    text = prep_for_tts(text).strip()
    if not text:
        return None
    steps = TTS_STEPS_FIRST if first else TTS_STEPS_FULL
    a = synthesize(text, steps, "first" if first else "chunk", save=save)
    a["cached"] = False
    a["steps"] = steps
    return a


def _join_audio(chunks):
    """Concatenates chunk waveforms (short gap between them) into one file. Returns (path, duration_s)."""
    if len(chunks) == 1 and chunks[0].get("path"):
        return chunks[0]["path"], chunks[0]["duration"]
    gap = np.zeros(int(CHUNK_GAP_S * TTS_SAMPLE_RATE), dtype=np.float32)
    parts = []
    for i, c in enumerate(chunks):
        if i:
            parts.append(gap)
        parts.append(np.asarray(c["waveform"], dtype=np.float32))
    wave = np.concatenate(parts)
    path = OUTPUT_DIR / f"full_{uuid.uuid4().hex[:8]}.wav"
    sf.write(str(path), wave, TTS_SAMPLE_RATE)
    return str(path), len(wave) / TTS_SAMPLE_RATE


def wav_to_b64(wave, sr=None):
    """float waveform -> base64 16-bit WAV, ready for the browser's decodeAudioData()."""
    sr = sr or TTS_SAMPLE_RATE
    pcm = (np.clip(np.asarray(wave, dtype=np.float32), -1.0, 1.0) * 32767).astype(np.int16)
    buf = io.BytesIO()
    sf.write(buf, pcm, sr, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode("ascii")


def _f(x, nd=4):
    """JSON-safe number: rounds, and turns NaN / inf / None into None."""
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return round(x, nd) if math.isfinite(x) else None


def _llm_reader(q, question, k):
    """Retrieval + streamed LLM in a thread. Every event is stamped with the time it happened, so TTS work in the
    consumer cannot delay the LLM timings (llama-server keeps generating while we synthesize)."""
    try:
        for ev in answer_stream(question, k):
            ev["_t"] = time.perf_counter()
            q.put(ev)
    except Exception as e:
        q.put({"type": "error", "error": e, "_t": time.perf_counter()})
    finally:
        q.put(None)


def run_pipeline(transcript, stream=True, k=None, t_start=None, emit=None):
    """Retrieval -> LLM -> TTS for one question (text). t_start lets the caller include ASR time.
    emit(event_dict) is called live (the widget uses it); without it the audio is written to disk instead."""
    k = k or RETRIEVAL_K
    web = emit is not None
    emit = emit or (lambda ev: None)
    if t_start is None:
        torch.cuda.reset_peak_memory_stats()
        t_start = time.perf_counter()
    ensure_server()

    def since(t=None):
        return (time.perf_counter() if t is None else t) - t_start

    q = queue.Queue()
    threading.Thread(target=_llm_reader, args=(q, transcript, k), daemon=True).start()

    sentences, chunks, per_sentence, marks = [], [], [], {}
    retrieved, final = None, None
    fallback = False

    def speak_and_emit(text, first, llm_at):
        tts_start = since()
        a = _speak(text, first=first, save=not web)
        if a is None:
            return None
        chunks.append(a)
        b64 = wav_to_b64(a["waveform"]) if web else None
        ready = since()                                  # the moment the chunk is handed over (after encoding)
        marks.setdefault("first_audio_ready", ready)
        rec = {"idx": len(chunks) - 1, "text": text, "llm_at_s": _f(llm_at), "tts_start_s": _f(tts_start),
               "waited_s": _f(max(0.0, tts_start - llm_at)), "tts_s": _f(a["time"]), "audio_dur_s": _f(a["duration"]),
               "rtf": _f(a["rtf"]) or 0.0, "steps": int(a.get("steps", 0)), "cached": bool(a["cached"]),
               "ready_at_s": _f(ready)}
        per_sentence.append(rec)
        emit({"type": "audio", "wav_b64": b64, **rec})
        return a

    try:
        while True:
            ev = q.get()
            if ev is None:
                break
            kind, at = ev["type"], since(ev["_t"])
            if kind == "error":
                raise ev["error"]
            elif kind == "retrieved":
                retrieved = ev
                marks["retrieval_done"] = at
                emit({"type": "retrieved", "at_s": _f(at), "retrieval_s": _f(ev["retrieval_s"]), "top_dense": _f(ev["top_dense"]),
                      "ids": [str(x["id"]) for x in ev["results"]]})
            elif kind == "first_token":
                marks["first_token"] = at
                emit({"type": "first_token", "at_s": _f(at), "ttft_s": _f(ev["t"])})
            elif kind == "sentence":
                sentences.append(ev["text"])
                marks.setdefault("first_sentence", at)
                emit({"type": "sentence", "idx": len(sentences) - 1, "text": ev["text"], "at_s": _f(at)})
                if stream:
                    speak_and_emit(ev["text"], first=not chunks, llm_at=at)
            elif kind == "done":
                final = ev
                marks["llm_done"] = at
    except (requests.RequestException, RuntimeError, torch.cuda.OutOfMemoryError) as e:
        print(f"LLM stage failed ({type(e).__name__}: {e}); using the fallback reply if nothing was produced.")
        free_cuda()
        fallback = True

    full_text = " ".join(sentences).strip()
    if not full_text:
        fallback, full_text = True, FALLBACK_REPLY
    if final is None:
        final = {"text": full_text, "total_s": 0.0, "ttft_s": 0.0, "first_sentence_s": 0.0, "new_tokens": 0,
                 "prompt_tokens": 0, "tok_s": 0.0, "prefill_s": 0.0, "server_s": 0.0}
        marks.setdefault("llm_done", since())

    if not chunks:
        # Single-pass mode (or nothing was spoken yet): one TTS call for the whole answer.
        speak_and_emit(full_text, first=False, llm_at=marks.get("llm_done", since()))

    refine_s, refined = 0.0, None
    # The optional refine pass produces a separate file, so it only runs outside the widget.
    if (not web and stream and TTS_REFINE_FINAL and not fallback and not is_idk(full_text)
            and TTS_STEPS_FIRST < TTS_STEPS_FULL):
        refined = synthesize(prep_for_tts(full_text), TTS_STEPS_FULL, "full")
        refine_s = refined["time"]

    if refined is not None:
        audio_path, audio_dur = refined["path"], refined["duration"]
    elif web:
        audio_path = None
        audio_dur = sum(c["duration"] for c in chunks) + CHUNK_GAP_S * (len(chunks) - 1)
    else:
        audio_path, audio_dur = _join_audio(chunks)

    tts_total = sum(c["time"] for c in chunks) + refine_s
    marks["all_done"] = since()
    total_s = marks["all_done"]
    retrieved = retrieved or {"results": [], "retrieval_s": 0.0, "top_dense": 0.0}

    return {
        "transcript": transcript,
        "first_phrase": sentences[0] if sentences else None,
        "full_response": full_text,
        "audio_path": audio_path,
        "first_audio_path": chunks[0].get("path") if (stream and len(chunks) >= 1) else None,
        "fallback": fallback,
        "idk": is_idk(full_text),
        "stream": stream,
        "time_sliced": stream,                     # kept for compatibility with the old notebook
        "asr_s": 0.0,
        "asr_detail": {},
        "marks": marks,                            # seconds since the request started, one entry per hand-over
        "per_sentence": per_sentence,
        "retrieval_s": retrieved["retrieval_s"],
        "top_dense": retrieved["top_dense"],
        "retrieved": retrieved["results"],
        "retrieved_ids": [x["id"] for x in retrieved["results"]],
        "llm_ttft_s": final["ttft_s"],
        "llm_first_sentence_s": final["first_sentence_s"],
        "llm_server_s": final.get("server_s", 0.0),
        "prefill_s": final.get("prefill_s", 0.0),
        "new_tokens": final["new_tokens"],
        "prompt_tokens": final["prompt_tokens"],
        "tok_s": final["tok_s"],
        "first_sentence_at_s": marks.get("first_sentence"),
        "first_audio_ready_s": marks.get("first_audio_ready", total_s),
        "tts_first_s": chunks[0]["time"],
        "first_audio_duration_s": chunks[0]["duration"],
        "tts_total_s": tts_total,
        "n_chunks": len(chunks),
        "refine_s": refine_s,
        "full_audio_duration_s": audio_dur,
        "full_tts_rtf": tts_total / audio_dur if audio_dur > 0 else float("nan"),
        "total_compute_s": total_s,
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
        "gpu_used_gb": gpu_used_gb(),
    }


def _attach_asr(r, asr, asr_at):
    r["asr_s"] = asr["total_s"]
    r["asr_detail"] = {"decode_s": asr["load_s"], "prep_s": asr["preprocessing_s"], "infer_s": asr["inference_s"]}
    r["marks"]["asr_done"] = asr_at


def run_s2s(audio_path, stream=True, time_sliced=None, k=None):
    """ASR -> retrieval -> LLM -> TTS from an audio file. `time_sliced` is the old name of `stream`."""
    if time_sliced is not None:
        stream = time_sliced
    torch.cuda.reset_peak_memory_stats()
    t_start = time.perf_counter()

    asr = transcribe(audio_path)
    asr_at = time.perf_counter() - t_start
    if not asr["transcript"]:
        raise ValueError("No speech was transcribed.")

    r = run_pipeline(asr["transcript"], stream=stream, k=k, t_start=t_start)
    _attach_asr(r, asr, asr_at)
    return r


def run_text_s2s(text, stream=True, k=None):
    """Same pipeline from typed text (no ASR)."""
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty question.")
    return run_pipeline(text, stream=stream, k=k)


def summarize(r):
    """JSON-safe summary of one run; the widget renders it."""
    d = r.get("asr_detail") or {}
    return {
        "stream": bool(r["stream"]), "idk": bool(r["idk"]), "fallback": bool(r["fallback"]),
        "full_response": r["full_response"],
        "marks": {k: _f(v) for k, v in r["marks"].items()},
        "asr": {"total_s": _f(r["asr_s"]), "decode_s": _f(d.get("decode_s")), "prep_s": _f(d.get("prep_s")), "infer_s": _f(d.get("infer_s"))},
        "retrieval_s": _f(r["retrieval_s"]), "top_dense": _f(r["top_dense"]), "ids": [str(i) for i in r["retrieved_ids"]],
        "llm": {"ttft_s": _f(r["llm_ttft_s"]), "first_sentence_s": _f(r["llm_first_sentence_s"]), "prefill_s": _f(r["prefill_s"]),
                "new_tokens": int(r["new_tokens"]), "prompt_tokens": int(r["prompt_tokens"]), "tok_s": _f(r["tok_s"], 2),
                "server_s": _f(r["llm_server_s"])},
        "tts": {"n_chunks": int(r["n_chunks"]), "total_s": _f(r["tts_total_s"]), "first_s": _f(r["tts_first_s"]),
                "first_audio_dur_s": _f(r["first_audio_duration_s"]), "rtf": _f(r["full_tts_rtf"], 3)},
        "total_s": _f(r["total_compute_s"]),
        "peak_vram_gb": _f(r["peak_vram_gb"], 2), "gpu_used_gb": _f(r["gpu_used_gb"], 2),
    }


_MILESTONES = [("asr_done", "ASR transcript ready"), ("retrieval_done", "Retrieval done"), ("first_token", "LLM first token"),
               ("first_sentence", "LLM first sentence complete"), ("llm_done", "LLM answer complete"),
               ("first_audio_ready", "FIRST SENTENCE AUDIO READY (TTFS)"), ("all_done", "Everything finished")]


def format_metrics(r):
    mode = "streaming (each sentence goes to TTS as it arrives)" if r["stream"] else "single-pass (full answer, then TTS)"
    m = r["marks"]
    lines = [f"Mode: {mode}", "Timeline (seconds since the request started; Δ = since the previous milestone):"]
    prev = 0.0
    for key, label in sorted([(k, l) for k, l in _MILESTONES if k in m], key=lambda kl: m[kl[0]]):
        lines.append(f"  {label:<36} at {m[key]:6.2f} s   Δ {m[key] - prev:+6.2f} s")
        prev = m[key]
    d = r.get("asr_detail") or {}
    if d:
        lines.append(f"ASR: {r['asr_s']:.2f} s (decode {d['decode_s'] * 1000:.0f} ms | features {d['prep_s'] * 1000:.0f} ms | model {d['infer_s'] * 1000:.0f} ms)")
    lines += [
        f"Retrieval: {r['retrieval_s'] * 1000:.0f} ms (top dense {r['top_dense']:.3f}) | chunks {r['retrieved_ids']}",
        f"LLM: first token {r['llm_ttft_s']:.2f} s | first sentence {r['llm_first_sentence_s']:.2f} s | "
        f"{r['new_tokens']} tokens @ {r['tok_s']:.1f} tok/s | prompt {r['prompt_tokens']} tok (prefill {r['prefill_s']:.2f} s)",
        f"TTS: {r['n_chunks']} chunk(s), {r['tts_total_s']:.2f} s total (RTF {r['full_tts_rtf']:.2f}) | first sentence synth {r['tts_first_s']:.2f} s",
        f"TIME TO FIRST SENTENCE (TTFS): {r['first_audio_ready_s']:.2f} s",
        f"Full audio duration: {r['full_audio_duration_s']:.2f} s | Total compute: {r['total_compute_s']:.2f} s",
        f"Peak VRAM (PyTorch models): {r['peak_vram_gb']:.2f} GB | GPU in use now (all processes): {r['gpu_used_gb']:.2f} GB",
    ]
    if r["idk"]:
        lines.append("Note: the assistant answered 'I don't know' (no relevant context found).")
    if r["fallback"]:
        lines.append("WARNING: fallback reply used (LLM failure or empty output).")
    return "\n".join(lines)


def speak_question(text, stream=True):
    """Type a question, hear and see the answer, and get the latency breakdown. No microphone needed."""
    r = run_text_s2s(text, stream=stream)
    print("Q:", r["transcript"])
    print("A:", r["full_response"])
    print("-" * 70)
    print(format_metrics(r))
    if r["first_audio_path"] and r["n_chunks"] > 1:
        print("\nFirst sentence (what streaming plays first):")
        display(Audio(r["first_audio_path"]))
    print("\nFull spoken answer:")
    display(Audio(r["audio_path"]))
    return r

IDK audio cached (5.52 s)


### Router, answer cache, numbers to words

The original `run_pipeline()` is not changed. It gets its answer from `answer_stream()` and its audio from `_speak()`; this cell wraps
those two functions:

* `answer_stream()`: if the router says the question is small talk, or the answer cache has seen the question, it yields the stored
  sentences straight away instead of retrieving and asking Gemma. Otherwise it is the original function, and the answer is remembered.
* `_speak()`: returns audio that already exists (router answers made at start-up, sentences said before) instead of synthesizing it
  again. Otherwise it is the original function.
* `split_ready()`: the original splitter, plus the early first chunk (`early_first_words`).
* `NUM2WORDS`: the original hook for turning digits into words before TTS, set to `convert_to_spoken` from the project's own `src/spoken.py`.

In [25]:
# wraps the original answer_stream() and _speak(); run_pipeline() itself is unchanged
try:
    from src.router import route_query
except ImportError:
    from src.router_latest import route_query
from src.spoken import convert_to_spoken
from src.fidel_fold import fold

if "_orig_answer_stream" not in globals():          # capture the ORIGINALS only once, so re-running this cell is safe
    _orig_answer_stream, _orig_speak = answer_stream, _speak
DIRECT = {"GREETING", "GREETING_SMALLTALK", "THANK_YOU", "GOODBYE", "META_IDENTITY"}
ROUTER_AUDIO, AUDIO_MEMO, ANSWER_CACHE = {}, {}, []

def _sentences(text):
    return [s.strip() for s in re.findall(r"[^።!?]+[።!?]?", text) if sanitize_to_ethiopic(s)]
def _audio(text, steps):
    a = synthesize(re.sub(r"\bAI\b", "ኤ አይ", convert_to_spoken(text)), steps, "premade", save=False)
    a["cached"], a["steps"] = True, steps
    return a
for _sample in ["ሰላም", "አመሰግናለሁ", "ቻው", "ማን ነህ?"]:              # router answers, made once now
    _r = route_query(_sample)
    for _s in _sentences(_r.get("direct_response") or ""):
        ROUTER_AUDIO.setdefault(_s, _audio(_s, TTS_STEPS_FULL))
print("router answers made at start-up:", len(ROUTER_AUDIO), "sentences")

def _key(text): return " ".join(re.findall(r"[ሀ-ፚ]+", fold(text)))
def _qvec(text): return model.encode(["query: " + normalize_query(text)], normalize_embeddings=True, convert_to_numpy=True)[0]
def _cache_lookup(question):
    key = _key(question)
    for c in ANSWER_CACHE:
        if c["key"] == key: return c, "exact"
    if ANSWER_CACHE:
        qv = _qvec(question); sims = [float(c["qv"] @ qv) for c in ANSWER_CACHE]; i = int(np.argmax(sims))
        if sims[i] >= IMPROVE["cache_min"]: return ANSWER_CACHE[i], f"similar ({sims[i]:.3f})"
    return None, None
LAST_SHORTCUT = {"why": None}
def _replay(sentences, why):
    """The same events the original generate_stream() yields, for an answer that is already known."""
    LAST_SHORTCUT["why"] = why
    t0 = time.perf_counter()
    yield {"type": "retrieved", "results": [], "retrieval_s": 0.0, "top_dense": 0.0}
    yield {"type": "first_token", "t": 0.0}
    for s in sentences: yield {"type": "sentence", "text": s, "t": time.perf_counter() - t0}
    yield {"type": "done", "text": " ".join(sentences), "total_s": time.perf_counter() - t0, "ttft_s": 0.0, "first_sentence_s": 0.0,
           "new_tokens": 0, "prompt_tokens": 0, "tok_s": 0.0, "prefill_s": 0.0, "server_s": 0.0}

def answer_stream(question, k=3, min_dense=None):
    LAST_SHORTCUT["why"] = None
    if IMPROVE["router"]:
        r = route_query(question)
        intent = str(getattr(r["intent"], "value", r["intent"]))
        if intent in DIRECT and r.get("direct_response"):
            yield from _replay(_sentences(r["direct_response"]), "router: " + intent); return
    if IMPROVE["answer_cache"]:
        hit, how = _cache_lookup(question)
        if hit: yield from _replay(hit["sentences"], "answer cache: " + how); return
    said, done = [], None
    for ev in _orig_answer_stream(question, k, min_dense):                # the original retrieval + Gemma
        if ev["type"] == "sentence": said.append(ev["text"])
        if ev["type"] == "done": done = ev
        yield ev
    if IMPROVE["answer_cache"] and said and done is not None:
        ANSWER_CACHE.append({"key": _key(question), "qv": _qvec(question), "sentences": said})

def _speak(text, first, save=True):
    known = (ROUTER_AUDIO.get(text) if IMPROVE["router"] else None) or (AUDIO_MEMO.get(text) if IMPROVE["answer_cache"] else None)
    if known is not None:
        a = dict(known); a.update(time=0.0, rtf=0.0, cached=True, steps=0); return a
    a = _orig_speak(text, first, save)                                     # the original TTS
    if a is not None and IMPROVE["answer_cache"]: AUDIO_MEMO[text] = a
    return a

if "_orig_split_ready" not in globals():
    _orig_split_ready = split_ready
def split_ready(buf, first):
    """The original sentence splitter. If the first phrase is still not finished after IMPROVE["early_first_words"] complete words,
    those words are sent to TTS now; the rest of the sentence follows as the next chunk."""
    phrases, rest, still_first = _orig_split_ready(buf, first)
    n = IMPROVE["early_first_words"]
    if n and still_first and not phrases:
        words = list(re.finditer(r"\S+\s+", rest))                 # words already followed by a space = complete
        if len(words) >= n and len(sanitize_to_ethiopic(rest[:words[n - 1].end()]).split()) >= n:
            cut = words[n - 1].end()
            return [rest[:cut]], rest[cut:], False
    return phrases, rest, still_first

def apply_improvements(**changes):
    """Change switches; restarts llama-server when the GPU or a server flag changes, then warms it up."""
    global NUM2WORDS
    IMPROVE.update(changes)
    NUM2WORDS = convert_to_spoken if IMPROVE["numbers_to_words"] else None
    # always restart: llama-server keeps whole old prompts in RAM, so a question asked again on the same server would skip
    # reading its prompt and look faster than it is
    start_server(); generate_full("ሰላም", max_tokens=8)
    ANSWER_CACHE.clear(); AUDIO_MEMO.clear()
    print("improvements:", IMPROVE, "| Gemma on GPU", llm_gpu())
apply_improvements()

router answers made at start-up: 12 sentences
Starting llama-server on port 36381 (GPU 1)...
llama-server ready in 4.0s
improvements: {'second_gpu': True, 'router': True, 'answer_cache': True, 'cache_min': 0.98, 'numbers_to_words': True, 'swa_full': False, 'early_first_words': 0} | Gemma on GPU 1


In [26]:
# Quick end-to-end check on the first answerable question (text in, speech out, with the full timeline):
_demo = speak_question([q for q in qs if q.get("answerable")][0]["question"])

Q: የሙከራ ጊዜ ከስንት የሥራ ቀናት መብለጥ የለበትም?
A: የሙከራ ጊዜው ሠራተኛው ሥራ ከጀመረበት ቀን አንስቶ ከስድሳ የሥራ ቀናት ሊበልጥ አይችልም።
----------------------------------------------------------------------
Mode: streaming (each sentence goes to TTS as it arrives)
Timeline (seconds since the request started; Δ = since the previous milestone):
  Retrieval done                       at   0.03 s   Δ  +0.03 s
  LLM first token                      at   0.60 s   Δ  +0.57 s
  LLM first sentence complete          at   1.38 s   Δ  +0.79 s
  LLM answer complete                  at   1.41 s   Δ  +0.02 s
  FIRST SENTENCE AUDIO READY (TTFS)    at   2.57 s   Δ  +1.16 s
  Everything finished                  at   2.57 s   Δ  +0.00 s
Retrieval: 23 ms (top dense 0.837) | chunks ['a011_c1', 'a027_c3', 'a064_c1']
LLM: first token 0.57 s | first sentence 1.35 s | 36 tokens @ 43.3 tok/s | prompt 1357 tok (prefill 0.56 s)
TTS: 1 chunk(s), 1.16 s total (RTF 0.11) | first sentence synth 1.16 s
TIME TO FIRST SENTENCE (TTFS): 2.57 s
Full audio dura

### GPU utilization

Shows what is loaded on each GPU and how busy each one is while a few questions are answered. The widget below has the same table live.

In [27]:
# GPU utilization: what runs on each GPU, and how busy each one is while a question is answered
def gpu_snapshot():
    """One reading per GPU: utilization, memory, and which process holds that memory."""
    def q(args):
        return subprocess.run(["nvidia-smi"] + args, capture_output=True, text=True, timeout=10).stdout.strip().splitlines()

    def num(x):
        try:
            return int(float(x))
        except ValueError:
            return 0

    gpus = {}
    for line in q(["--query-gpu=index,uuid,name,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"]):
        idx, uid, name, util, used, total = [x.strip() for x in line.split(",")]
        gpus[uid] = {"index": num(idx), "name": name, "util": num(util), "mem_used_mib": num(used), "mem_total_mib": num(total), "procs": []}
    srv = globals().get("server_proc")
    srv_pid = srv.pid if srv is not None and srv.poll() is None else None
    for line in q(["--query-compute-apps=gpu_uuid,pid,process_name,used_memory", "--format=csv,noheader,nounits"]):
        p = [x.strip() for x in line.split(",")]
        if len(p) < 4 or p[0] not in gpus:
            continue
        pid = num(p[1])
        role = ("llama-server (Gemma 4)" if pid == srv_pid
                else "this notebook (ASR + TTS + retrieval)" if pid == os.getpid()
                else os.path.basename(p[2]))
        gpus[p[0]]["procs"].append({"pid": pid, "role": role, "mem_mib": num(p[3])})
    return sorted(gpus.values(), key=lambda g: g["index"])


def show_gpus():
    for g in gpu_snapshot():
        print(f"GPU {g['index']}  {g['name']}   util {g['util']:3d} %   memory {g['mem_used_mib'] / 1024:5.1f} / {g['mem_total_mib'] / 1024:.1f} GB")
        for p in g["procs"] or [{"role": "(nothing)", "mem_mib": 0}]:
            print(f"      {p['role']:<40} {p['mem_mib'] / 1024:5.1f} GB")


def profile_gpus(questions):
    """Answers each question end to end while sampling both GPUs, then prints peak / mean utilization per GPU."""
    seen, stop = {}, threading.Event()

    def sampler():
        while not stop.is_set():
            try:
                for g in gpu_snapshot():
                    s = seen.setdefault(g["index"], {"name": g["name"], "util": [], "mem": 0})
                    s["util"].append(g["util"])
                    s["mem"] = max(s["mem"], g["mem_used_mib"])
            except Exception:
                pass
            time.sleep(0.05)

    th = threading.Thread(target=sampler, daemon=True)
    th.start()
    t0 = time.perf_counter()
    for text in questions:
        run_text_s2s(text)
    stop.set()
    th.join()
    print(f"{len(questions)} question(s) answered in {time.perf_counter() - t0:.1f} s; samples taken while they ran:")
    for i, s in sorted(seen.items()):
        u = s["util"] or [0]
        print(f"  GPU {i} {s['name']}: utilization peak {max(u)} %, mean {sum(u) / len(u):.0f} % ({len(u)} samples), memory peak {s['mem'] / 1024:.1f} GB")


print("Where things run:")
print("  GPU 0: ASR (Hohe CTC, fp16), OmniVoice TTS (fp16) and the retrieval embedding model, all inside this notebook's process")
print(f"  GPU {llm_gpu()}: Gemma 4 E4B (Q4_K_M) inside llama-server" + ("" if llm_gpu() else "  (second_gpu is off or only one GPU: it shares GPU 0)"))
print("\nNow:")
show_gpus()
print()
profile_gpus([q["question"] for q in qs if q.get("answerable")][1:4])   # new questions, so the answer cache cannot skip the work


Where things run:
  GPU 0: ASR (Hohe CTC, fp16), OmniVoice TTS (fp16) and the retrieval embedding model, all inside this notebook's process
  GPU 1: Gemma 4 E4B (Q4_K_M) inside llama-server

Now:
GPU 0  Tesla T4   util  24 %   memory   4.8 / 15.0 GB
      this notebook (ASR + TTS + retrieval)      4.8 GB
GPU 1  Tesla T4   util   0 %   memory   3.3 / 15.0 GB
      llama-server (Gemma 4)                     3.3 GB

3 question(s) answered in 11.5 s; samples taken while they ran:
  GPU 0 Tesla T4: utilization peak 100 %, mean 54 % (96 samples), memory peak 4.8 GB
  GPU 1 Tesla T4: utilization peak 100 %, mean 38 % (96 samples), memory peak 3.3 GB


## 10. Voice widget (raw HTML / CSS / JS, no Gradio)

Two cells:

1. **Backend**: registers two callbacks the page calls and a small web server that serves the page and those callbacks. `notebook.s2s_start` receives the recording and starts the pipeline in a background thread. `notebook.s2s_poll` is a *long poll*: it returns the instant new events (transcript, sentence text, sentence audio, timings) exist, so there is no fixed polling delay.
2. **Widget**: the HTML, CSS and JavaScript (edit the string in that cell to restyle it), then a tunnel link. Open the link in a new tab, press **Record**, speak Amharic, press **Stop**.

The page opens in its own tab because the notebook's output frame blocks the microphone. If the browser asks, allow the microphone for the link's site. The second cell keeps running while the widget is live (the link stops working when it ends); interrupt it to close the widget. Re-run it for a fresh link. Cells below it only run after you interrupt it.

In [28]:
class _KaggleOutput:            # [Kaggle] stands in for google.colab.output: keeps the callbacks the web adapter below serves
    callbacks = {}
    def register_callback(self, name, fn): self.callbacks[name] = fn
output = _KaggleOutput()
from IPython.display import HTML, JSON, display

_PIPE_LOCK = threading.Lock()      # one request at a time on the GPU
_JOBS = {}


class _Job:
    """Event list the pipeline thread writes to and the browser's long poll reads from."""

    def __init__(self):
        self.events, self.done = [], False
        self.cv = threading.Condition()

    def push(self, ev):
        with self.cv:
            self.events.append(ev)
            self.cv.notify_all()

    def finish(self):
        with self.cv:
            self.done = True
            self.cv.notify_all()


def _run_job(job, wav_b64, stream, t_start):
    try:
        with _PIPE_LOCK:
            torch.cuda.reset_peak_memory_stats()
            asr = transcribe_wav_bytes(base64.b64decode(wav_b64))
            asr_at = time.perf_counter() - t_start
            if not asr["transcript"]:
                job.push({"type": "error", "message": "No speech was transcribed. Try again."})
                return
            job.push({"type": "asr", "transcript": asr["transcript"], "at_s": _f(asr_at), "asr_s": _f(asr["total_s"]),
                      "decode_s": _f(asr["load_s"]), "prep_s": _f(asr["preprocessing_s"]), "infer_s": _f(asr["inference_s"])})
            r = run_pipeline(asr["transcript"], stream=stream, t_start=t_start, emit=job.push)
            _attach_asr(r, asr, asr_at)
            job.push({"type": "done", "summary": summarize(r)})
    except Exception as e:
        job.push({"type": "error", "message": f"{type(e).__name__}: {e}"})
    finally:
        job.finish()


def s2s_start(wav_b64, stream=True):
    """Called by the page when Stop is pressed. Returns at once; the work runs in a thread."""
    t_start = time.perf_counter()                 # t = 0 of the server-side timeline
    job_id = uuid.uuid4().hex[:10]
    _JOBS[job_id] = job = _Job()
    for old_id in list(_JOBS)[:-5]:               # keep only the last few jobs
        _JOBS.pop(old_id, None)
    threading.Thread(target=_run_job, args=(job, wav_b64, bool(stream), t_start), daemon=True).start()
    return JSON({"job": job_id})


def s2s_poll(job_id, cursor):
    """Long poll: waits up to LONG_POLL_S for events after `cursor` and returns them as soon as there are any."""
    job = _JOBS.get(job_id)
    if job is None:
        return JSON({"events": [{"type": "error", "message": "unknown job"}], "cursor": cursor, "done": True})
    with job.cv:
        if len(job.events) <= cursor and not job.done:
            job.cv.wait(timeout=LONG_POLL_S)
        events = job.events[cursor:]
        return JSON({"events": events, "cursor": cursor + len(events), "done": job.done})


output.register_callback("notebook.s2s_start", s2s_start)
output.register_callback("notebook.s2s_poll", s2s_poll)
print("Callbacks registered: notebook.s2s_start, notebook.s2s_poll")

# ---- Web adapter: Kaggle has no google.colab callback channel, so the same two callbacks are served over plain HTTP ----
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

_httpd = None


def _widget_page():
    """The widget as a standalone page (WIDGET_HTML is defined in the next cell and read on every request)."""
    return ("<!doctype html><html><head><meta charset='utf-8'>"
            "<meta name='viewport' content='width=device-width,initial-scale=1'>"
            "<title>Amharic Voice Assistant</title></head>"
            "<body style='margin:16px;background:#fff'>" + WIDGET_HTML + "</body></html>")


class _WidgetHandler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        pass

    def _send(self, code, body, ctype):
        self.send_response(code)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        path = self.path.split("?")[0]
        if path in ("/", "/index.html"):
            self._send(200, _widget_page().encode("utf-8"), "text/html; charset=utf-8")
        elif path == "/health":
            self._send(200, b"ok", "text/plain")
        elif path == "/gpu":
            try:
                self._send(200, json.dumps(gpu_snapshot()).encode("utf-8"), "application/json")
            except Exception as e:
                self._send(500, json.dumps({"error": f"{type(e).__name__}: {e}"}).encode("utf-8"), "application/json")
        else:
            self._send(404, b"not found", "text/plain")

    def do_POST(self):
        path = self.path.split("?")[0]
        try:
            n = int(self.headers.get("Content-Length") or 0)
            args = json.loads(self.rfile.read(n) or b"[]")
            fn = output.callbacks.get(path[len("/rpc/"):]) if path.startswith("/rpc/") else None
            if fn is None:
                self._send(404, b'{"error": "unknown callback"}', "application/json")
                return
            res = fn(*args)
            res = res.data if hasattr(res, "data") else res        # IPython JSON(...) -> plain dict
            self._send(200, json.dumps(res).encode("utf-8"), "application/json")
        except (BrokenPipeError, ConnectionResetError):
            pass
        except Exception as e:
            self._send(500, json.dumps({"error": f"{type(e).__name__}: {e}"}).encode("utf-8"), "application/json")


def start_widget_server():
    """Starts the local web server for the widget (stops an earlier one first). Returns the port."""
    global _httpd
    if _httpd is not None:
        _httpd.shutdown()
        _httpd.server_close()
    ThreadingHTTPServer.daemon_threads = True
    _httpd = ThreadingHTTPServer(("127.0.0.1", _free_port()), _WidgetHandler)
    threading.Thread(target=_httpd.serve_forever, daemon=True).start()
    return _httpd.server_address[1]


Callbacks registered: notebook.s2s_start, notebook.s2s_poll


In [29]:
# ==============================================================================
# Voice widget: raw HTML + CSS + JS (recorder from the Hohe ASR notebook, extended with streaming
# playback and latency panels). Edit freely; re-run this cell to get a fresh link.
# ==============================================================================
WIDGET_HTML = r"""
<style>
#s2s{font-family:sans-serif;max-width:800px;border:2px solid #4CAF50;border-radius:10px;padding:15px;background:#fff;color:#222;box-sizing:border-box}
#s2s h4{margin:0 0 10px;text-align:center}
#s2s .ctl{display:flex;flex-wrap:wrap;gap:10px;align-items:center;justify-content:center}
#s2s button{color:#fff;padding:10px 20px;border:none;border-radius:5px;cursor:pointer;font-size:14px}
#s2s button:disabled{opacity:.5;cursor:default}
#s2s #startBtn{background-color:#4CAF50}
#s2s #stopBtn{background-color:#f44336}
#s2s #replayBtn{background-color:#607d8b}
#s2s #status{margin:10px 0;color:#555;font-size:14px;text-align:center}
#s2s .cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(108px,1fr));gap:8px;margin:10px 0}
#s2s .card{border:1px solid #ddd;border-radius:8px;padding:8px 4px;text-align:center;background:#fafafa}
#s2s .card.hl{border-color:#4CAF50;background:#e8f5e9}
#s2s .card .v{font-size:20px;font-weight:bold}
#s2s .card .l{font-size:12px;color:#333}
#s2s .card .s{font-size:11px;color:#888;min-height:13px}
#s2s .lbl{font-size:12px;color:#777;margin-top:10px}
#s2s .box{border:1px solid #ddd;border-radius:6px;padding:8px;margin:4px 0;min-height:20px;font-size:15px;white-space:pre-wrap}
#s2s .wrap{overflow-x:auto}
#s2s table{border-collapse:collapse;width:100%;font-size:12px;margin:4px 0}
#s2s th,#s2s td{border-bottom:1px solid #eee;padding:4px 6px;text-align:left;vertical-align:top}
#s2s th{background:#f5f5f5}
#s2s td.n{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}
#s2s tr.key td{background:#e8f5e9;font-weight:bold}
</style>

<div id="s2s">
    <h4>🎙️ Amharic Voice Assistant</h4>
    <div class="ctl">
        <button id="startBtn">Record</button>
        <button id="stopBtn" disabled>Stop</button>
        <button id="replayBtn" disabled>Replay answer</button>
        <label style="font-size:13px"><input type="checkbox" id="streamChk" __STREAM_CHECKED__> Streaming (speak each sentence as it arrives)</label>
    </div>
    <p id="status">Status: Ready</p>

    <div class="cards" id="cards"></div>

    <div class="lbl">You said (ASR)</div>
    <div class="box" id="transcript"></div>
    <div class="lbl">Assistant</div>
    <div class="box" id="response"></div>

    <div class="lbl">Pipeline timeline: how each stage hands over to the next (server clock, starts when the audio reaches Python)</div>
    <div class="wrap"><table id="timeline"></table></div>

    <div class="lbl">Browser side (clock starts at the Stop click)</div>
    <div class="wrap"><table id="browser"></table></div>

    <div class="lbl">Per sentence (TTS speed)</div>
    <div class="wrap"><table id="sentences"></table></div>

    <div class="lbl">GPUs (live, refreshed every 2 s)</div>
    <div class="wrap"><table id="gpus"></table></div>
</div>

<script>
(function () {
    const GAP = __GAP__;                       // seconds of silence between sentence chunks (CHUNK_GAP_S)
    const $ = (id) => document.getElementById(id);
    const esc = (s) => String(s).replace(/[&<>"]/g, (c) => ({'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;'}[c]));
    const ok  = (x) => x !== null && x !== undefined && !isNaN(x);
    const fx  = (x, n) => ok(x) ? Number(x).toFixed(n) : '—';
    const sec = (x) => ok(x) ? Number(x).toFixed(2) + ' s' : '—';
    const ms  = (x) => ok(x) ? Math.round(x * 1000) + ' ms' : '—';
    const dlt = (x) => ok(x) ? '+' + Number(x).toFixed(2) + ' s' : '—';

    // calls a Python callback: over HTTP on Kaggle (this page is served by the notebook), through Colab's channel on Colab
    const rpc = async (name, args) => {
        if (window.google && google.colab && google.colab.kernel) return google.colab.kernel.invokeFunction(name, args, {});
        const r = await fetch('rpc/' + name, { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify(args) });
        if (!r.ok) throw new Error('HTTP ' + r.status);
        return { data: { 'application/json': await r.json() } };
    };

    let audioContext, mediaStream, recProcessor;
    let audioBuffers = [];
    let playCtx = null, playHead = 0, active = [], played = [], chain = Promise.resolve();
    let runId = 0, S = null, T = {};

    // ---------- WAV encoder (same as the original recorder) ----------
    function encodeWAV(samples, sampleRate) {
        let buffer = new ArrayBuffer(44 + samples.length * 2);
        let view = new DataView(buffer);
        for (let i = 0; i < 4; i++) view.setUint8(i, "RIFF".charCodeAt(i));
        view.setUint32(4, 36 + samples.length * 2, true);
        for (let i = 0; i < 4; i++) view.setUint8(8 + i, "WAVE".charCodeAt(i));
        for (let i = 0; i < 4; i++) view.setUint8(12 + i, "fmt ".charCodeAt(i));
        view.setUint32(16, 16, true);
        view.setUint16(20, 1, true);
        view.setUint16(22, 1, true);
        view.setUint32(24, sampleRate, true);
        view.setUint32(28, sampleRate * 2, true);
        view.setUint16(32, 2, true);
        view.setUint16(34, 16, true);
        for (let i = 0; i < 4; i++) view.setUint8(36 + i, "data".charCodeAt(i));
        view.setUint32(40, samples.length * 2, true);
        let index = 44;
        for (let i = 0; i < samples.length; i++) {
            let s = Math.max(-1, Math.min(1, samples[i]));
            view.setInt16(index, s < 0 ? s * 0x8000 : s * 0x7FFF, true);
            index += 2;
        }
        return new Blob([view], { type: 'audio/wav' });
    }

    const setStatus = (t) => { $('status').innerText = t; };

    // ---------- state ----------
    function reset() {
        S = { stream: $('streamChk').checked, asr: null, retr: null, ft: null, sents: [], audio: [], done: null, error: null };
        T = {};
        played = [];
        chain = Promise.resolve();
        playHead = 0;
        $('transcript').textContent = '';
        $('response').textContent = '';
        $('replayBtn').disabled = true;
        render();
    }

    function stopPlayback() {
        active.forEach((s) => { try { s.stop(); } catch (e) {} });
        active = [];
    }

    // ---------- rendering ----------
    function render() {
        const a = S.asr, r = S.retr, f = S.ft, d = S.done, stream = S.stream;
        const first = S.audio[0];
        const llmEnd = stream ? (S.sents[0] ? S.sents[0].at_s : null) : (d ? d.marks.llm_done : null);
        const audioAt = first ? first.ready_at_s : null;
        const asrAt = a ? a.at_s : null, retrAt = r ? r.at_s : null;
        const sub = (x, y) => (ok(x) && ok(y)) ? x - y : null;
        const up = (ok(T.started) && ok(T.stop)) ? (T.started - T.stop) / 1000 : null;
        const bTtfs = (ok(T.heard) && ok(T.stop)) ? (T.heard - T.stop) / 1000 : null;
        const deliver = (ok(bTtfs) && ok(up) && ok(audioAt)) ? Math.max(0, bTtfs - up - audioAt) : null;

        // summary cards: ASR + Retrieval + LLM + TTS (+ hand-overs) add up to the server TTFS
        const card = (v, l, s, hl) => '<div class="card' + (hl ? ' hl' : '') + '"><div class="v">' + v + '</div><div class="l">' + l + '</div><div class="s">' + s + '</div></div>';
        $('cards').innerHTML =
            card(sec(asrAt), 'ASR', a ? 'model ' + sec(a.infer_s) : '') +
            card(sec(sub(retrAt, asrAt)), 'Retrieval', r ? 'compute ' + ms(r.retrieval_s) : '') +
            card(sec(sub(llmEnd, retrAt)), stream ? 'LLM → 1st sentence' : 'LLM → full answer',
                 (d && ok(d.llm.tok_s)) ? fx(d.llm.tok_s, 1) + ' tok/s' : (f ? '1st token ' + sec(f.ttft_s) : '')) +
            card(sec(sub(audioAt, llmEnd)), 'TTS → 1st audio', first ? 'synth ' + sec(first.tts_s) + ' · RTF ' + fx(first.rtf, 2) : '') +
            card(sec(audioAt), 'TTFS (server)', 'request → 1st sentence audio', true) +
            card(sec(bTtfs), 'TTFS (browser)', 'Stop click → heard', true);

        // server timeline
        const rows = [['Audio reached Python', 0, null, '']];
        let prev = 0;
        const add = (label, at, detail, key) => {
            rows.push([label, at, ok(at) ? at - prev : null, detail, key]);
            if (ok(at)) prev = at;
        };
        add('ASR transcript ready', asrAt, a ? 'decode ' + ms(a.decode_s) + ' · features ' + ms(a.prep_s) + ' · model ' + ms(a.infer_s) : '');
        add('Retrieval done', retrAt, r ? 'compute ' + ms(r.retrieval_s) + ' · top dense ' + fx(r.top_dense, 3) + ' · chunks ' + esc(r.ids.join(', ')) : '');
        add('LLM first token', f ? f.at_s : null,
            f ? 'request + prompt prefill ' + sec(f.ttft_s) + (d ? ' · prompt ' + d.llm.prompt_tokens + ' tok · server prefill ' + sec(d.llm.prefill_s) : '') : '');
        add(stream ? 'LLM first sentence complete' : 'LLM full answer complete', llmEnd,
            d ? d.llm.new_tokens + ' tokens @ ' + fx(d.llm.tok_s, 1) + ' tok/s (whole answer)' : '');
        add('FIRST SENTENCE AUDIO READY (TTFS)', audioAt,
            first ? 'TTS synth ' + sec(first.tts_s) + ' (' + first.steps + ' steps' + (first.cached ? ', cached' : '') + ') · audio ' + sec(first.audio_dur_s) + ' · RTF ' + fx(first.rtf, 2) : '', true);
        add('Everything finished', d ? d.total_s : null, d ? d.tts.n_chunks + ' audio chunk(s) · TTS total ' + sec(d.tts.total_s) : '');
        $('timeline').innerHTML = '<tr><th>Milestone</th><th>at</th><th>Δ from previous</th><th>detail</th></tr>' +
            rows.map((x) => '<tr' + (x[4] ? ' class="key"' : '') + '><td>' + x[0] + '</td><td class="n">' + sec(x[1]) + '</td><td class="n">' + dlt(x[2]) + '</td><td>' + x[3] + '</td></tr>').join('');

        // browser side
        const brow = [
            ['Encode WAV + upload to Python', sec(up), false],
            ['Server: request → first sentence audio ready', sec(audioAt), false],
            ['Delivery back to the browser (poll + decode + schedule)', sec(deliver), false],
            ['TTFS in the browser: Stop click → first sentence audible', sec(bTtfs), true]
        ];
        $('browser').innerHTML = brow.map((x) => '<tr' + (x[2] ? ' class="key"' : '') + '><td>' + x[0] + '</td><td class="n">' + x[1] + '</td></tr>').join('');

        // per sentence
        let srows = S.audio.map((e, i) =>
            '<tr><td class="n">' + (i + 1) + '</td><td>' + esc((e.text || '').slice(0, 48)) + '</td>' +
            '<td class="n">' + sec(e.llm_at_s) + '</td><td class="n">' + sec(e.waited_s) + '</td>' +
            '<td class="n">' + sec(e.tts_s) + '</td><td class="n">' + sec(e.audio_dur_s) + '</td><td class="n">' + fx(e.rtf, 2) + '</td>' +
            '<td class="n">' + sec(e.ready_at_s) + '</td><td class="n">' + (ok(e.heardMs) ? sec((e.heardMs - T.stop) / 1000) : '…') + '</td></tr>');
        if (stream) {
            for (let i = S.audio.length; i < S.sents.length; i++) {
                srows.push('<tr><td class="n">' + (i + 1) + '</td><td>' + esc(S.sents[i].text.slice(0, 48)) + '</td><td class="n">' + sec(S.sents[i].at_s) +
                           '</td><td colspan="6">synthesizing…</td></tr>');
            }
        }
        $('sentences').innerHTML = '<tr><th>#</th><th>sentence</th><th>LLM ready at</th><th>waited for TTS</th><th>TTS time</th><th>audio length</th><th>RTF</th><th>audio ready at</th><th>heard at (from Stop)</th></tr>' + srows.join('');
    }

    // ---------- audio playback: decode each sentence and queue it gapless ----------
    function b64ToBuf(b64) {
        const bin = atob(b64), u8 = new Uint8Array(bin.length);
        for (let i = 0; i < bin.length; i++) u8[i] = bin.charCodeAt(i);
        return u8.buffer;
    }

    function startSource(buf) {
        const src = playCtx.createBufferSource();
        src.buffer = buf;
        src.connect(playCtx.destination);
        src.start(playHead);
        active.push(src);
        src.onended = () => { active = active.filter((x) => x !== src); };
    }

    function schedule(buf, ev) {
        const now = playCtx.currentTime;
        if (playHead < now + 0.03) playHead = now + 0.03;
        ev.heardMs = performance.now() + (playHead - now) * 1000;
        if (T.heard === undefined) T.heard = ev.heardMs;
        startSource(buf);
        played.push(buf);
        playHead += buf.duration + GAP;
        $('replayBtn').disabled = false;
        render();
    }

    function enqueue(ev, myRun) {
        S.audio.push(ev);
        chain = chain.then(async () => {
            if (myRun !== runId) return;
            try {
                const buf = await playCtx.decodeAudioData(b64ToBuf(ev.wav_b64));
                ev.wav_b64 = null;
                if (myRun !== runId) return;
                schedule(buf, ev);
            } catch (err) {
                setStatus('❌ Audio decode error: ' + err.message);
            }
        });
    }

    $('replayBtn').onclick = () => {
        if (!played.length || !playCtx) return;
        stopPlayback();
        playHead = playCtx.currentTime + 0.05;
        played.forEach((b) => { startSource(b); playHead += b.duration + GAP; });
    };

    // ---------- events coming back from Python ----------
    function handle(ev, myRun) {
        switch (ev.type) {
            case 'asr':
                S.asr = ev;
                $('transcript').textContent = ev.transcript;
                setStatus('🧠 Transcribed. Retrieving and generating...');
                break;
            case 'retrieved': S.retr = ev; break;
            case 'first_token': S.ft = ev; break;
            case 'sentence':
                S.sents.push(ev);
                $('response').textContent += (S.sents.length > 1 ? ' ' : '') + ev.text;
                setStatus('🔊 Speaking...');
                break;
            case 'audio': enqueue(ev, myRun); break;
            case 'done': S.done = ev.summary; break;
            case 'error':
                S.error = ev.message;
                setStatus('❌ ' + ev.message);
                break;
        }
        render();
    }

    async function runJob(base64Audio) {
        const myRun = ++runId;
        setStatus('⏳ Processing in Python...');
        try {
            const res = await rpc('notebook.s2s_start', [base64Audio, S.stream]);
            T.started = performance.now();
            const job = res.data['application/json'].job;
            let cursor = 0;
            while (myRun === runId) {
                const r = await rpc('notebook.s2s_poll', [job, cursor]);
                const d = r.data['application/json'];
                cursor = d.cursor;
                for (const ev of d.events) handle(ev, myRun);
                if (d.done) break;
            }
            if (myRun === runId) {
                await chain;
                if (!S.error) setStatus('✅ Completed! Ready for next input.');
            }
        } catch (err) {
            setStatus('❌ Python Error: ' + (err && err.message ? err.message : err) + ' (see the browser console)');
            console.error('Colab Kernel Error:', err);
        }
    }

    // ---------- microphone ----------
    $('startBtn').onclick = async () => {
        try {
            // the playback context must be created/resumed inside a click handler (browser autoplay rules)
            if (!playCtx) playCtx = new (window.AudioContext || window.webkitAudioContext)();
            await playCtx.resume();
            runId++;                 // abandon any answer that is still streaming in
            stopPlayback();
            reset();

            mediaStream = await navigator.mediaDevices.getUserMedia({ audio: true });
            audioContext = new (window.AudioContext || window.webkitAudioContext)({ sampleRate: 16000 });
            let source = audioContext.createMediaStreamSource(mediaStream);

            recProcessor = audioContext.createScriptProcessor(4096, 1, 1);
            audioBuffers = [];
            recProcessor.onaudioprocess = (e) => {
                let inputBuffer = e.inputBuffer.getChannelData(0);
                audioBuffers.push(new Float32Array(inputBuffer));
            };

            source.connect(recProcessor);
            recProcessor.connect(audioContext.destination);

            $('startBtn').disabled = true;
            $('stopBtn').disabled = false;
            setStatus('🔴 Recording... Speak in Amharic');
        } catch (err) {
            const hint = (err && err.name === 'NotAllowedError') ? ' (allow the microphone for this site in the browser address bar)'
                       : (!navigator.mediaDevices ? ' (the microphone needs a secure https page)' : '');
            setStatus('❌ Mic Error: ' + err.message + hint);
        }
    };

    $('stopBtn').onclick = async () => {
        T.stop = performance.now();      // t = 0 for the browser-side timings
        try {
            recProcessor.disconnect();
            mediaStream.getTracks().forEach((track) => track.stop());
            audioContext.close();

            let totalLength = audioBuffers.reduce((acc, buf) => acc + buf.length, 0);
            let pcmData = new Float32Array(totalLength);
            let offset = 0;
            for (let buf of audioBuffers) {
                pcmData.set(buf, offset);
                offset += buf.length;
            }

            let wavBlob = encodeWAV(pcmData, 16000);
            let reader = new FileReader();
            reader.readAsDataURL(wavBlob);
            reader.onloadend = () => {
                runJob(reader.result.split(',')[1]);
            };
        } catch (err) {
            setStatus('❌ Encoding Error: ' + err.message);
        }

        $('startBtn').disabled = false;
        $('stopBtn').disabled = true;
    };

    // live GPU panel (served by the notebook at /gpu; shows "not available" where that does not exist, e.g. on Colab)
    const gpuPanel = async () => {
        try {
            const r = await fetch('gpu', { cache: 'no-store' });
            if (!r.ok) throw new Error('HTTP ' + r.status);
            const g = await r.json();
            $('gpus').innerHTML = '<tr><th>GPU</th><th>utilization</th><th>memory</th><th>running on it</th></tr>' +
                g.map((x) => '<tr><td>' + x.index + ' · ' + esc(x.name) + '</td><td class="n">' + x.util + ' %</td><td class="n">' +
                    (x.mem_used_mib / 1024).toFixed(1) + ' / ' + (x.mem_total_mib / 1024).toFixed(1) + ' GB</td><td>' +
                    (x.procs.map((p) => esc(p.role) + ' · ' + (p.mem_mib / 1024).toFixed(1) + ' GB').join('<br>') || '—') + '</td></tr>').join('');
        } catch (e) {
            $('gpus').innerHTML = '<tr><td>GPU stats not available here</td></tr>';
        }
    };
    gpuPanel();
    setInterval(gpuPanel, 2000);

    reset();
})();
</script>
"""

WIDGET_HTML = (WIDGET_HTML
               .replace("__GAP__", str(CHUNK_GAP_S))
               .replace("__STREAM_CHECKED__", "checked" if WIDGET_STREAMING_DEFAULT else ""))

# ---- Serve the widget and open it in its own browser tab -------------------------------------------------------
# The notebook's output frame on Kaggle blocks the microphone ("permission denied"), so the page is served by a small web
# server inside this notebook and reached through a public https tunnel link. The page itself is the widget above.
import urllib.request

_tunnel = {"proc": None}


def stop_widget():
    """Closes the tunnel and the local web server."""
    global _httpd
    p = _tunnel.get("proc")
    if p is not None and p.poll() is None:
        p.terminate()
    _tunnel["proc"] = None
    if _httpd is not None:
        _httpd.shutdown()
        _httpd.server_close()
        _httpd = None


def _cloudflared_cmd(port):
    path = WRITABLE_DIR / "cloudflared"
    if not path.exists():
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", str(path))
    os.chmod(path, 0o755)
    # http2 instead of the default QUIC (UDP): Kaggle blocks UDP, which leaves the link printed but dead ("Error 1033")
    return [str(path), "tunnel", "--no-autoupdate", "--protocol", "http2", "--edge-ip-version", "4",
            "--url", f"http://127.0.0.1:{port}"]


def _ssh_cmd(port):
    return ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "UserKnownHostsFile=/dev/null", "-o", "ServerAliveInterval=30",
            "-R", f"80:127.0.0.1:{port}", "nokey@localhost.run"]


def open_tunnel(port, timeout=60):
    """Public https link to the local widget server. Tries Cloudflare first, then localhost.run.
    A link is only returned once it really answers (the link is printed by the tunnel before it is connected)."""
    import collections
    for name, make_cmd, pattern in (("cloudflared", _cloudflared_cmd, r"https://[a-z0-9-]+\.trycloudflare\.com"),
                                    ("localhost.run", _ssh_cmd, r"https://[a-z0-9]+\.lhr\.life")):
        try:
            proc = subprocess.Popen(make_cmd(port), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        except Exception as e:
            print(f"  {name}: could not start ({type(e).__name__}: {e})")
            continue
        found, tail, state = queue.Queue(), collections.deque(maxlen=15), {"registered": False}

        def _read(p=proc, pat=pattern, out=found):
            for line in p.stdout:                      # keeps draining so the tunnel never blocks on a full pipe
                tail.append(line.rstrip())
                if "Registered tunnel connection" in line:
                    state["registered"] = True
                m = re.search(pat, line)
                if m and "//api." not in m.group(0):
                    out.put(m.group(0))

        threading.Thread(target=_read, daemon=True).start()
        try:
            url = found.get(timeout=timeout)
        except queue.Empty:
            proc.terminate()
            print(f"  {name}: no link within {timeout} s. Last output:\n    " + "\n    ".join(tail))
            continue
        reachable = False
        for _ in range(60):                            # wait until the link answers (or the tunnel reports it is connected)
            if proc.poll() is not None:
                break
            try:
                if requests.get(url + "/health", timeout=5).status_code == 200:
                    reachable = True
                    break
            except requests.RequestException:
                pass
            if state["registered"]:
                time.sleep(3)
                reachable = True
                break
            time.sleep(1)
        if reachable:
            _tunnel["proc"] = proc
            return url
        if proc.poll() is None:
            proc.terminate()
        print(f"  {name}: the link {url} never started answering. Last output:\n    " + "\n    ".join(tail))
    raise RuntimeError("Could not open a working tunnel link. Check that Internet is on for this notebook; "
                       "the output above shows what the tunnel programs reported.")


stop_widget()
atexit.register(stop_widget)
WIDGET_PORT = start_widget_server()
WIDGET_URL = open_tunnel(WIDGET_PORT)
print("Voice widget link:", WIDGET_URL)
display(HTML(
    '<div style="font-family:sans-serif;border:2px solid #4CAF50;border-radius:10px;padding:14px;max-width:800px">'
    '<b>🎙️ Amharic Voice Assistant is ready.</b><br><br>'
    f'<a href="{WIDGET_URL}" target="_blank" rel="noopener" style="font-size:16px">Open the voice widget in a new tab</a>'
    f'<br><small>{WIDGET_URL}</small><br><br>'
    '<small>Allow the microphone when the browser asks. The link works only while this cell is running and is reachable by anyone '
    'who has it. Press Stop (interrupt) on this cell to close it.</small></div>'))

# The link only works while the tunnel and web server are alive. If this cell simply ended, "Run all" would go straight on to the
# last cell, which stops them (the link then shows Cloudflare "Error 1033"), and a saved/committed run would end the same way.
# So the cell keeps running until you interrupt it.


WIDGET_KEEP_ALIVE = False
if WIDGET_KEEP_ALIVE:
    print("Widget is live. Leave this cell running while you use it; interrupt it to close the widget.")
    try:
        while True:
            time.sleep(5)
            _p = _tunnel.get("proc")
            if _p is None or _p.poll() is not None:
                print("The tunnel closed. Run this cell again for a new link.")
                break
    except KeyboardInterrupt:
        pass
    finally:
        stop_widget()
        print("Widget stopped.")


Voice widget link: https://newsletters-supplemental-process-listed.trycloudflare.com


## 11. Switching quantization and stopping

If answers are worse than expected, set `LLAMA_QUANT = "Q8_0"` in the configuration cell, rerun the download and start-server cells, and try the widget again (no need to redisplay it). Q8_0 is slower but close to full quality and still fits the T4 next to ASR, TTS and retrieval.

Run the last cell when you are done (the server also stops automatically when the runtime ends).

In [30]:
# ---- Test: synthesize a sample question as audio (no microphone needed) --------------------
# Picks a real answerable question from the labeled set and speaks it, so the next cell can
# feed it through the full pipeline exactly like a real recording would.
TEST_QUESTION_TEXT = [q for q in qs if q.get("answerable")][0]["question"]
print("Question text:", TEST_QUESTION_TEXT)

_test_q_audio = synthesize(TEST_QUESTION_TEXT, TTS_STEPS_FULL, "test_question")
TEST_QUESTION_AUDIO_PATH = _test_q_audio["path"]
print(f"Synthesized question audio: {TEST_QUESTION_AUDIO_PATH} ({_test_q_audio['duration']:.2f}s)")
display(Audio(TEST_QUESTION_AUDIO_PATH))


Question text: የሙከራ ጊዜ ከስንት የሥራ ቀናት መብለጥ የለበትም?
Synthesized question audio: /tmp/s2s_0v9ojky3/test_question_15d5415e.wav (5.72s)


In [31]:
# ---- Test: run that synthesized question through the full pipeline ------------------------
# run_s2s() exercises ASR too (unlike speak_question(), which skips straight to retrieval) -
# this is the closest thing to a real spoken question without actually using a microphone.
_test_result = run_s2s(TEST_QUESTION_AUDIO_PATH)
print("ASR transcript:", _test_result["transcript"])
print("Answer:", _test_result["full_response"])
print("-" * 70)
print(format_metrics(_test_result))
display(Audio(_test_result["audio_path"]))


ASR transcript: የሙከራ ጊዜ ከስንት የስራ ቀናት መብለጥ የለበትም
Answer: የሙከራ ጊዜው ሠራተኛው ሥራ ከጀመረበት ቀን አንስቶ ከስድሳ የሥራ ቀናት ሊበልጥ አይችልም።
----------------------------------------------------------------------
Mode: streaming (each sentence goes to TTS as it arrives)
Timeline (seconds since the request started; Δ = since the previous milestone):
  ASR transcript ready                 at  16.02 s   Δ +16.02 s
  Retrieval done                       at  16.02 s   Δ  +0.00 s
  LLM first token                      at  16.02 s   Δ  +0.00 s
  LLM first sentence complete          at  16.02 s   Δ  +0.00 s
  LLM answer complete                  at  16.02 s   Δ  +0.00 s
  FIRST SENTENCE AUDIO READY (TTFS)    at  16.02 s   Δ  +0.00 s
  Everything finished                  at  16.02 s   Δ  +0.00 s
ASR: 16.02 s (decode 15576 ms | features 28 ms | model 417 ms)
Retrieval: 0 ms (top dense 0.000) | chunks []
LLM: first token 0.00 s | first sentence 0.00 s | 0 tokens @ 0.0 tok/s | prompt 0 tok (prefill 0.00 s)
TTS: 1 chunk(s), 

In [32]:
stop_server()
stop_widget()
